# Dictionary Discovery Workflow v4 - Complete Implementation

## Overview
Systematic dictionary-based topic discovery and model training workflow with structured file organization.

### Checkpoints:
- **CHECKPOINT 0**: Initial Setup → Create folders, load config
- **CHECKPOINT 1**: Text Processing → Chunk corpus
- **CHECKPOINT 2**: Vocabulary Building → Build vocab from chunks
- **CHECKPOINT 3**: Dictionary Expansion → Expand keywords (⚠️ MANUAL CURATION REQUIRED)
- **CHECKPOINT 4**: Topic Vectors → Build weighted topic vectors
- **CHECKPOINT 5**: Chunk Scoring → Score & classify by confidence
- **CHECKPOINT 6**: Training Data Prep → Create train/val splits
- **CHECKPOINT 7**: Model Training → Train BERTJE
- **CHECKPOINT 8**: BERTJE Labeling → Label corpus with trained model
- **CHECKPOINT 9**: Visualizations → Generate clustering & performance plots

### Folder Structure:
```
workflow_data/{ModelType}-{Topic}_{Date}_{Version}/
  ├── config/
  ├── Dictionary/
  │   └── Dictionary_suggestions/
  ├── Model_finetuning/
  ├── Cosine_labeling/
  ├── Bertje_labeling/
  ├── Visuals/
  └── Other_data/
```

---
# CHECKPOINT 0: Initial Setup
---

In [2]:
# ============================================================
# CELL 0.1: IMPORTS
# ============================================================
import os
import re
import json
import hashlib
import shutil
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

# NLTK
import nltk
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# ML libraries
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sentence_transformers import SentenceTransformer

# Suppress warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

print("✓ All imports successful")

✓ All imports successful


In [3]:
# ============================================================
# CONFIGURATION WITH CORPUS FILTERING
# ============================================================

CONFIG = {
    # =====================
    # WORKFLOW METADATA
    # =====================
    "workflow": {
        # Model type: "Pretrained" or "Finetuned_{source}"
        "model_type": "Pretrained_Slavery",
        
        # Topic: "Slavery", "Policy", "Slavery-Policy", etc.
        "topic": "Slavery",
        
        # Version (None for auto-increment)
        "version": None,
    },

    # =====================
    # PATHS
    # =====================
    "paths": {
        "corpus_dir": "Slavery_text",
        "dictionary_excel": "dutch_slavery_legacy_dictionary.xlsx",
        "workflow_base": "workflow_data",
        "pretrained_model_path": "workflow_data\\Finetuned_Slavery-policy-Slavery-policy_10.30.25_v1\\Model_finetuning",
    },
    
    # =====================
    # CORPUS FILTERING (NEW SECTION)
    # =====================
    "corpus_filter": {
        # Enable/disable filtering
        "enabled": False,
        
        # Year filtering options:
        # Option 1: Specific years as a list
        "years": [2022],
        
        # Option 2: Year range (overrides "years" if set)
        # Set to None to disable, or use tuple like (2015, 2023)
        "year_range": None,  # e.g., (2015, 2023) for 2015 through 2023
        
        # Option 3: Only documents after/before certain year
        "year_min": None,  # e.g., 2000 for documents from 2000 onwards
        "year_max": None,  # e.g., 2023 for documents up to 2023
        
        # Document type filtering
        # List of document types to include (None = all types)
        "doc_types": ["beleidsnotas", "jaarplannen","besluiten","jaarverslagen"],  # e.g., ["policy", "report", "legislation"]
        
        # Filename pattern matching (uses regex)
        # List of patterns - documents matching ANY pattern are included
        "filename_patterns": None,  # e.g., [".*gemeente.*", ".*ministry.*"]
        
        # Exclude patterns (uses regex)
        # Documents matching ANY exclude pattern are skipped
        "exclude_patterns": None,  # e.g., [".*draft.*", ".*concept.*"]
    },
    
    # =====================
    # MODEL SETTINGS
    # =====================
    "model": {
        "base_model_name": "NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers",
        "use_pretrained": False,
    },
    
    # =====================
    # DICTIONARY SETTINGS
    # =====================
    "dictionary": {
        "use_excel": True,
        "topic_column": "topic",
        "keyword_column": "keyword",
        "sheet_name": 0,
        "default_topics": {
            "Historical slavery": ["slavernij", "tot-slaaf-gemaakte", "dwangarbeid", "zweep"],
            "Colonialism": ["kolonie", "koloniaal", "voc", "wic", "exploitatie"],
            "Modern racism& inequality": ["racisme", "discriminatie", "ongelijkheid"],
        },
    },
    
    # =====================
    # TEXT PROCESSING
    # =====================
    "chunking": {
        "sentences_per_chunk": 10,
        "min_sentences_to_keep": 3,
        "drop_likely_english": True,
        "remove_stopwords": True,
        "use_stemming": False,
    },
    
    "tokenize": {
        "lower": True,
        "keep_hyphen": True,
        "min_len": 2,
        "max_len": 30,
        "pattern": r"[0-9A-Za-zÀ-ÖØ-öø-ÿ\-]+",
    },
    
    # =====================
    # VOCABULARY SETTINGS
    # =====================
    "vocab": {
        "min_df": 5,
        "max_vocab": 50000,
    },
    
    # =====================
    # EXPANSION SETTINGS
    # =====================
    "expand": {
        "k_nearest": 50,
        "topN_per_topic": 300,
        "min_cosine": 0.55,
    },
    
    # =====================
    # SCORING SETTINGS
    # =====================
    "scoring": {
        "use_sif": True,
        "sif_a": 1e-3,
        "high_confidence_score": 0.50,
        "high_confidence_margin": 0.05,
        "low_confidence_score": 0.40,
        "low_confidence_margin": 0.02,
    },
    
    # =====================
    # TRAINING SETTINGS
    # =====================
    "training": {
        "num_epochs": 5,
        "batch_size_train": 16,
        "batch_size_eval": 32,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "dataset_option": "option4",
    },
}

print("✓ Configuration loaded")
print(f"\nWorkflow: {CONFIG['workflow']['model_type']}-{CONFIG['workflow']['topic']}")

# Display active filters
if CONFIG['corpus_filter']['enabled']:
    print("\n📂 Corpus Filtering ENABLED:")
    
    # Year filters
    if CONFIG['corpus_filter']['year_range']:
        print(f"  - Year range: {CONFIG['corpus_filter']['year_range'][0]} to {CONFIG['corpus_filter']['year_range'][1]}")
    elif CONFIG['corpus_filter']['years']:
        print(f"  - Specific years: {CONFIG['corpus_filter']['years']}")
    elif CONFIG['corpus_filter']['year_min'] or CONFIG['corpus_filter']['year_max']:
        if CONFIG['corpus_filter']['year_min']:
            print(f"  - Year minimum: {CONFIG['corpus_filter']['year_min']}")
        if CONFIG['corpus_filter']['year_max']:
            print(f"  - Year maximum: {CONFIG['corpus_filter']['year_max']}")
    
    # Document type filters
    if CONFIG['corpus_filter']['doc_types']:
        print(f"  - Document types: {CONFIG['corpus_filter']['doc_types']}")
    
    # Pattern filters
    if CONFIG['corpus_filter']['filename_patterns']:
        print(f"  - Include patterns: {CONFIG['corpus_filter']['filename_patterns']}")
    if CONFIG['corpus_filter']['exclude_patterns']:
        print(f"  - Exclude patterns: {CONFIG['corpus_filter']['exclude_patterns']}")
else:
    print("\n📂 Corpus Filtering DISABLED - loading all documents")

✓ Configuration loaded

Workflow: Pretrained_Slavery-Slavery

📂 Corpus Filtering DISABLED - loading all documents


In [4]:
# ============================================================
# CELL 0.3: FILE SYSTEM UTILITIES
# ============================================================

class WorkflowFileSystem:
    """Manages structured folder system for workflow data."""
    
    def __init__(self, config):
        self.config = config
        self.root = None
        self.folders = {}
    
    def create_workflow_folder(self):
        """Create main workflow folder with subfolders."""
        model_type = self.config["workflow"]["model_type"]
        topic = self.config["workflow"]["topic"]
        date = datetime.now().strftime("%m.%d.%y")
        
        version = self.config["workflow"]["version"]
        if version is None:
            version = self._get_next_version(model_type, topic, date)
        
        folder_name = f"{model_type}-{topic}_{date}_{version}"
        base_dir = self.config["paths"]["workflow_base"]
        self.root = Path(base_dir) / folder_name
        self.root.mkdir(parents=True, exist_ok=True)
        
        subfolder_names = [
            "config",
            "Dictionary",
            "Dictionary/Dictionary_suggestions",
            "Model_finetuning",
            "Cosine_labeling",
            "Bertje_labeling",
            "Visuals",
            "Other_data",
        ]
        
        for subfolder in subfolder_names:
            path = self.root / subfolder
            path.mkdir(parents=True, exist_ok=True)
            key = subfolder.split("/")[-1]
            self.folders[key] = path
        
        self.folders["Dictionary"] = self.root / "Dictionary"
        
        print(f"\n{'='*60}")
        print("WORKFLOW FOLDER CREATED")
        print(f"{'='*60}")
        print(f"Location: {self.root}")
        print(f"\nSubfolders:")
        for name in subfolder_names:
            print(f"  ✓ {name}/")
        
        return self.root
    
    def _get_next_version(self, model_type, topic, date):
        """Auto-increment version number."""
        base_dir = Path(self.config["paths"]["workflow_base"])
        if not base_dir.exists():
            return "v1"
        
        prefix = f"{model_type}-{topic}_{date}_v"
        existing = [d.name for d in base_dir.iterdir() if d.is_dir() and d.name.startswith(prefix)]
        
        if not existing:
            return "v1"
        
        versions = []
        for folder in existing:
            try:
                version_str = folder.split("_v")[-1]
                versions.append(int(version_str.replace("v", "")))
            except:
                continue
        
        if versions:
            return f"v{max(versions) + 1}"
        return "v1"
    
    def load_existing_workflow(self, folder_path):
        """Load existing workflow folder."""
        self.root = Path(folder_path)
        if not self.root.exists():
            raise ValueError(f"Workflow folder not found: {folder_path}")
        
        subfolder_names = [
            "config", "Dictionary", "Dictionary_suggestions",
            "Model_finetuning", "Cosine_labeling", "Bertje_labeling",
            "Visuals", "Other_data"
        ]
        
        for name in subfolder_names:
            if name == "Dictionary_suggestions":
                path = self.root / "Dictionary" / name
            else:
                path = self.root / name
            if path.exists():
                self.folders[name] = path
        
        print(f"✓ Loaded existing workflow: {self.root.name}")
        return self.root
    
    def save_config(self, checkpoint_name=None):
        """Save CONFIG to config folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"config_{checkpoint_name}_{timestamp}.json" if checkpoint_name else f"config_{timestamp}.json"
        config_path = self.folders["config"] / filename
        
        config_data = {
            "metadata": {
                "timestamp": timestamp,
                "checkpoint": checkpoint_name,
                "workflow_folder": str(self.root),
            },
            "config": self.config
        }
        
        with open(config_path, 'w', encoding='utf-8') as f:
            json.dump(config_data, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Config saved: {config_path.name}")
        return config_path
    
    def save_data(self, data, filename, folder_key, file_format="csv"):
        """Save data to specific folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        full_filename = f"{filename}.{file_format}"
        filepath = folder / full_filename
        
        if file_format == "csv":
            if not isinstance(data, pd.DataFrame):
                raise ValueError("CSV format requires DataFrame")
            data.to_csv(filepath, index=False, encoding='utf-8')
        elif file_format == "json":
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
        elif file_format == "npy":
            np.save(filepath, data, allow_pickle=True)
        else:
            raise ValueError(f"Unsupported format: {file_format}")
        
        print(f"✓ Saved: {folder_key}/{full_filename}")
        return filepath
    
    def copy_file_to_folder(self, source_path, folder_key, new_name=None):
        """Copy external file to workflow folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        source = Path(source_path)
        if not source.exists():
            raise FileNotFoundError(f"Source file not found: {source_path}")
        
        dest_name = new_name if new_name else source.name
        dest_path = folder / dest_name
        
        shutil.copy2(source, dest_path)
        print(f"✓ Copied: {source.name} → {folder_key}/{dest_name}")
        return dest_path

print("✓ WorkflowFileSystem class defined")

✓ WorkflowFileSystem class defined


In [5]:
# ============================================================
# CELL 0.4: CREATE OR LOAD WORKFLOW
# ============================================================

# Choose one:
CREATE_NEW = False  # Set to False to load existing
EXISTING_FOLDER = r"C:\Users\Home\policy-analysis\workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1"  # Set path if loading existing

fs = WorkflowFileSystem(CONFIG)

if CREATE_NEW:
    workflow_root = fs.create_workflow_folder()
    fs.save_config("initial_setup")
else:
    if EXISTING_FOLDER is None:
        raise ValueError("EXISTING_FOLDER must be set when CREATE_NEW=False")
    workflow_root = fs.load_existing_workflow(EXISTING_FOLDER)

print(f"\n✓ Workflow initialized: {workflow_root}")

✓ Loaded existing workflow: Pretrained_Slavery-Slavery_10.30.25_v1

✓ Workflow initialized: C:\Users\Home\policy-analysis\workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1


In [93]:
# ============================================================
# CELL 0.5: LOAD DICTIONARY
# ============================================================

def load_dictionary_from_excel(excel_path, config):
    """Load topics and keywords from Excel."""
    if not Path(excel_path).exists():
        print(f"⚠ Excel not found: {excel_path}")
        return config["dictionary"]["default_topics"]
    
    try:
        df = pd.read_excel(excel_path, sheet_name=config["dictionary"]["sheet_name"])
        topic_col = config["dictionary"]["topic_column"]
        keyword_col = config["dictionary"]["keyword_column"]
        
        if topic_col not in df.columns or keyword_col not in df.columns:
            return config["dictionary"]["default_topics"]
        
        topics_dict = {}
        for topic, group in df.groupby(topic_col):
            keywords = group[keyword_col].dropna().str.strip().tolist()
            if keywords:
                topics_dict[topic] = keywords
        
        print(f"✓ Loaded from Excel: {len(topics_dict)} topics, {len(df)} keywords")
        return topics_dict
    except Exception as e:
        print(f"⚠ Error: {e}")
        return config["dictionary"]["default_topics"]

if CONFIG["dictionary"]["use_excel"]:
    topics = load_dictionary_from_excel(CONFIG["paths"]["dictionary_excel"], CONFIG)
    CONFIG["topics"] = topics
    if Path(CONFIG["paths"]["dictionary_excel"]).exists():
        fs.copy_file_to_folder(
            CONFIG["paths"]["dictionary_excel"],
            "Dictionary",
            "input_dictionary.xlsx"
        )
else:
    CONFIG["topics"] = CONFIG["dictionary"]["default_topics"]

print(f"\n{'='*60}")
print("TOPICS LOADED")
print(f"{'='*60}")
for topic, keywords in CONFIG["topics"].items():
    print(f"  {topic}: {len(keywords)} keywords")

fs.save_config("with_dictionary")

✓ Loaded from Excel: 3 topics, 55 keywords
✓ Copied: dutch_slavery_legacy_dictionary.xlsx → Dictionary/input_dictionary.xlsx

TOPICS LOADED
  Colonialism: 19 keywords
  Historical Slavery: 18 keywords
  Modern Racism & Inequality: 18 keywords
✓ Config saved: config_with_dictionary_20251030_170853.json


WindowsPath('workflow_data/Pretrained_Slavery-Slavery_10.30.25_v1/config/config_with_dictionary_20251030_170853.json')

In [94]:
# ============================================================
# CELL 3.1: LOAD SENTENCE TRANSFORMER MODEL
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 3 START - LOADING VOCABULARY & MODEL")
print(f"{'='*60}")


print(f"\n{'='*60}")
print("LOADING SENTENCE TRANSFORMER MODEL")
print(f"{'='*60}")

if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
    model_path = CONFIG['paths']['pretrained_model_path']
    if Path(model_path).exists():
        st_model = SentenceTransformer(model_path)
        print(f"✓ Loaded pretrained model from: {model_path}")
    else:
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"⚠ Pretrained path not found, using base model")
else:
    st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
    print(f"✓ Loaded base model: {CONFIG['model']['base_model_name']}")

def st_embed(texts: list, batch_size: int = 256) -> np.ndarray:
    return st_model.encode(
        texts, 
        batch_size=batch_size, 
        show_progress_bar=False, 
        normalize_embeddings=True
    )

print(f"\n✓ Model ready")
print(f"  Max sequence length: {st_model.max_seq_length}")
print(f"  Embedding dimension: {st_model.get_sentence_embedding_dimension()}")



CHECKPOINT 3 START - LOADING VOCABULARY & MODEL

LOADING SENTENCE TRANSFORMER MODEL
✓ Loaded base model: NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers

✓ Model ready
  Max sequence length: 128
  Embedding dimension: 768


✅ **CHECKPOINT 0 COMPLETE** - Folder structure created, dictionary loaded

---
# CHECKPOINT 1: Text Processing
---

Chunks corpus into sentence-based segments with cleaning.

In [95]:
# ============================================================
# CELL 1.1: TEXT CLEANING UTILITIES
# ============================================================

stemmer = SnowballStemmer("dutch")

nltk_stopwords = set(stopwords.words('dutch')) | set(stopwords.words('english'))
custom_stopwords = set([
    "de","het","een","en","van","in","op","met","voor","tegen","zonder","bij",
    "naar","tot","uit","door","aan","om","te","als","ook","maar","want","dus",
    "of","dan","nog","wel","zijn","is","was","waren","worden","hebben","heeft",
    "had","doet","doen","al","alle","meer","minder","veel","weinig","binnen",
    "buiten","tussen","onder","boven","over","na","achter","naast","sinds",
    "tijdens","zoals","ik","jij","hij","zij","wij","jullie","u","je","ze",
    "dit","dat","die","deze","welke","ons","hun","hem","haar","bijlage",
    "bijlagen","inleiding","samenvatting","conclusie","conclusies","jaar","jaren",
])
ALL_STOPWORDS = nltk_stopwords | custom_stopwords

ENGLISH_HINTS = set("the and of to in that is for on with as by from at it this be are were was has have will would can could should".split())
DUTCH_HINTS = set("de het een en van voor met op aan te is zijn worden was waren niet bij in over uit door naar tot als ook om".split())

def likely_english_sentence(s: str) -> bool:
    if not CONFIG["chunking"]["drop_likely_english"]:
        return False
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", s.lower())
    if not tokens:
        return False
    e = sum(t in ENGLISH_HINTS for t in tokens)
    d = sum(t in DUTCH_HINTS for t in tokens)
    return e > max(2, d + 1)

def remove_stopwords_and_numbers(text: str) -> str:
    if pd.isna(text):
        return ""
    tokens = re.findall(r"\b\w+\b", text.lower())
    filtered = [tok for tok in tokens if tok not in ALL_STOPWORDS and not tok.isdigit()]
    return " ".join(filtered)

def stem_text(text: str) -> str:
    tokens = re.findall(r"\b\w+\b", text.lower())
    return " ".join(stemmer.stem(w) for w in tokens)

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def short_file_hash(path: str, n=8) -> str:
    return hashlib.sha1(path.encode("utf-8", errors="ignore")).hexdigest()[:n]

def make_chunk_uid(file_path: str, chunk_idx: int) -> str:
    return f"{short_file_hash(file_path)}:{chunk_idx:05d}"

print("✓ Text cleaning utilities ready")

✓ Text cleaning utilities ready


In [96]:
# ============================================================
# METADATA EXTRACTION FROM NESTED FOLDER STRUCTURE
# ============================================================

def extract_path_metadata(file_path: str, corpus_dir: Path) -> dict:
    """
    Extract metadata from nested folder structure.
    
    Expected structure: corpus_dir/doc_type/year/document_folder/file.txt
    Example: Policyarchive_text/convenanten/2019/document-name/document-name.2.txt
    
    Args:
        file_path: Full path to the document file
        corpus_dir: Base directory of the corpus
    
    Returns:
        Dictionary with doc_type, year, document_folder, and filename
        - doc_type: The type of document (e.g., 'convenanten', 'kamerstukken')
        - year: The year extracted from the path (if parseable)
        - document_folder: The parent folder containing the document
        - filename: Just the filename
        - relative_path: Path relative to corpus_dir
    """
    try:
        # Get the path relative to the corpus directory
        # This removes the corpus_dir prefix so we can analyze the structure
        rel_path = Path(file_path).relative_to(corpus_dir)
        parts = rel_path.parts  # Split path into components
        
        # Initialize metadata dictionary with None values
        metadata = {
            'doc_type': None,           # Changed from 'category'
            'year': None,
            'document_folder': None,
            'filename': rel_path.name,
            'relative_path': str(rel_path)
        }
        
        # Extract doc_type from first level of folder structure
        # Example: if path is "convenanten/2019/doc/file.txt", doc_type = "convenanten"
        if len(parts) >= 2:
            metadata['doc_type'] = parts[0]  # Changed from 'category'
        
        # Extract year from second level (if it's a 4-digit number)
        if len(parts) >= 3:
            try:
                year_candidate = parts[1]
                # Check if it's a valid 4-digit year
                if year_candidate.isdigit() and len(year_candidate) == 4:
                    metadata['year'] = int(year_candidate)
            except:
                pass  # If parsing fails, year stays None
        
        # Extract document folder (the parent folder of the file)
        if len(parts) >= 4:
            metadata['document_folder'] = parts[-2]  # Second to last is document folder
        elif len(parts) == 3:
            metadata['document_folder'] = parts[-2]  # If only 3 levels, still use parent folder
        
        return metadata
        
    except Exception as e:
        # Fallback if path parsing fails - return basic info only
        return {
            'doc_type': None,
            'year': None,
            'document_folder': None,
            'filename': Path(file_path).name,
            'relative_path': str(Path(file_path).name)
        }

In [97]:
# ============================================================
# DOCUMENT FILTERING LOGIC
# ============================================================

def should_include_document(file_path: str, corpus_dir: Path, filter_config: dict) -> tuple:
    """
    Determine if a document should be included based on filter settings.
    
    This function checks the nested folder structure and filename against
    all configured filters to decide if a document should be processed.
    
    Args:
        file_path: Full path to the document file
        corpus_dir: Base corpus directory path
        filter_config: The corpus_filter section from CONFIG
        
    Returns:
        Tuple of (should_include: bool, reason: str, metadata: dict)
        - should_include: True if document passes all filters
        - reason: Explanation of why document was included/excluded
        - metadata: Document metadata extracted from path
    """
    # If filtering is disabled in config, include everything
    if not filter_config.get('enabled', False):
        metadata = extract_path_metadata(file_path, corpus_dir)
        return True, "filtering disabled", metadata
    
    # Extract metadata from the file path
    metadata = extract_path_metadata(file_path, corpus_dir)
    
    # Get relative path for pattern matching
    rel_path = metadata['relative_path']
    
    # ===== STEP 1: CHECK EXCLUDE PATTERNS FIRST =====
    # These take highest priority - if path matches any exclude pattern, skip it
    exclude_patterns = filter_config.get('exclude_patterns')
    if exclude_patterns:
        for pattern in exclude_patterns:
            if re.search(pattern, rel_path, re.IGNORECASE):
                return False, f"matches exclude pattern: {pattern}", metadata
    
    # ===== STEP 2: CHECK DOC_TYPE FILTERS =====
    # Changed from category to doc_type
    
    # Check if doc_type is in exclude list
    exclude_doc_types = filter_config.get('exclude_doc_types')  # Changed from 'exclude_categories'
    if exclude_doc_types and metadata['doc_type'] in exclude_doc_types:
        return False, f"doc_type '{metadata['doc_type']}' is excluded", metadata
    
    # Check if doc_type is in include list (if specified)
    doc_types = filter_config.get('doc_types')  # Changed from 'categories'
    if doc_types:
        if metadata['doc_type'] is None:
            # If doc_type is required but not found, exclude document
            if filter_config.get('require_doc_type', False):  # Changed from 'require_category'
                return False, "no doc_type found in path", metadata
        elif metadata['doc_type'] not in doc_types:
            return False, f"doc_type '{metadata['doc_type']}' not in allowed doc_types", metadata
    
    # ===== STEP 3: CHECK YEAR FILTERS =====
    year = metadata['year']
    
    # Check if year is required but missing
    if filter_config.get('require_year', False) and year is None:
        return False, "year required but not found in path", metadata
    
    # If year exists, check against year filters
    if year is not None:
        # Check year range (highest priority if specified)
        year_range = filter_config.get('year_range')
        if year_range:
            if not (year_range[0] <= year <= year_range[1]):
                return False, f"year {year} outside range {year_range}", metadata
        
        # Check specific years list
        elif filter_config.get('years'):
            if year not in filter_config['years']:
                return False, f"year {year} not in allowed years", metadata
        
        # Check min/max years
        else:
            year_min = filter_config.get('year_min')
            year_max = filter_config.get('year_max')
            
            if year_min and year < year_min:
                return False, f"year {year} before minimum {year_min}", metadata
            if year_max and year > year_max:
                return False, f"year {year} after maximum {year_max}", metadata
    
    # ===== STEP 4: CHECK DOCUMENT FOLDER FILTERS =====
    document_folders = filter_config.get('document_folders')
    if document_folders:
        if metadata['document_folder'] is None:
            return False, "no document folder found in path", metadata
        if metadata['document_folder'] not in document_folders:
            return False, f"document folder '{metadata['document_folder']}' not in allowed folders", metadata
    
    # ===== STEP 5: CHECK FILENAME PATTERNS =====
    filename_patterns = filter_config.get('filename_patterns')
    if filename_patterns:
        matched = False
        for pattern in filename_patterns:
            if re.search(pattern, metadata['filename'], re.IGNORECASE):
                matched = True
                break
        if not matched:
            return False, "filename doesn't match any include pattern", metadata
    
    # If we've made it through all filters, include the document
    return True, "passed all filters", metadata

In [98]:
# ============================================================
# MAIN CORPUS PROCESSING WITH FILTERING
# ============================================================

print(f"\n{'='*60}")
print("PROCESSING CORPUS (WITH FILTERING)")
print(f"{'='*60}")

# Initialize list to store all chunks from all documents
all_chunks = []
corpus_dir = Path(CONFIG["paths"]["corpus_dir"])

# Check if corpus directory exists
if not corpus_dir.exists():
    print(f"⚠ Corpus directory not found: {corpus_dir}")
else:
    # Recursively find all .txt files in nested folders
    # The ** pattern means "look in all subdirectories at any depth"
    all_doc_files = list(corpus_dir.glob("**/*.txt"))
    print(f"\nFound {len(all_doc_files)} total documents in nested folders")
    
    # Get filter configuration
    filter_config = CONFIG.get("corpus_filter", {"enabled": False})
    
    # ===== DISPLAY FILTER SETTINGS =====
    if filter_config.get('enabled', False):
        print("\n🔍 Applying filters...")
        
        # Display doc_type filters (changed from categories)
        print(f"  Doc types: {filter_config.get('doc_types', 'All')}")
        print(f"  Exclude doc types: {filter_config.get('exclude_doc_types', 'None')}")
        
        # Display year filters
        if filter_config.get('year_range'):
            print(f"  Year range: {filter_config['year_range'][0]} - {filter_config['year_range'][1]}")
        elif filter_config.get('years'):
            print(f"  Specific years: {filter_config['years']}")
        elif filter_config.get('year_min') or filter_config.get('year_max'):
            if filter_config.get('year_min'):
                print(f"  Year minimum: {filter_config['year_min']}")
            if filter_config.get('year_max'):
                print(f"  Year maximum: {filter_config['year_max']}")
        
        # Display other filters
        if filter_config.get('document_folders'):
            print(f"  Document folders: {filter_config['document_folders']}")
        if filter_config.get('filename_patterns'):
            print(f"  Filename patterns: {filter_config['filename_patterns']}")
        if filter_config.get('exclude_patterns'):
            print(f"  Exclude patterns: {filter_config['exclude_patterns']}")
    
    # ===== APPLY FILTERS TO ALL DOCUMENTS =====
    doc_files = []  # Documents that pass filters
    skipped_docs = []  # Documents that don't pass filters
    
    for doc_path in all_doc_files:
        include, reason, metadata = should_include_document(str(doc_path), corpus_dir, filter_config)
        if include:
            doc_files.append(doc_path)
        else:
            skipped_docs.append((doc_path.name, reason))
    
    # Display filtering results
    print(f"\n✅ {len(doc_files)} documents passed filters")
    if skipped_docs:
        print(f"⏭️  {len(skipped_docs)} documents skipped")
        # Show first 10 skipped documents as examples
        if len(skipped_docs) <= 10:
            print("\nSkipped documents:")
            for filename, reason in skipped_docs[:10]:
                print(f"  - {filename}: {reason}")
    
    # ===== ANALYZE FILTERED CORPUS STRUCTURE =====
    # Count documents by doc_type and year to show overview
    doc_types = {}  # Changed from categories
    years = {}
    
    for doc_path in doc_files:
        _, _, metadata = should_include_document(str(doc_path), corpus_dir, filter_config)
        
        # Count doc_types
        if metadata['doc_type']:
            doc_types[metadata['doc_type']] = doc_types.get(metadata['doc_type'], 0) + 1
        
        # Count years
        if metadata['year']:
            years[metadata['year']] = years.get(metadata['year'], 0) + 1
    
    # Display doc_type distribution (changed from categories)
    if doc_types:
        print(f"\n📁 Document types in filtered corpus:")
        for doc_type, count in sorted(doc_types.items()):
            print(f"  - {doc_type}: {count} documents")
    
    # Display year distribution
    if years:
        print(f"\n📅 Years in filtered corpus:")
        for year, count in sorted(years.items()):
            print(f"  - {year}: {count} documents")
    
    # ===== CHUNK ALL FILTERED DOCUMENTS =====
    print(f"\n{'='*60}")
    print("CHUNKING DOCUMENTS")
    print(f"{'='*60}\n")
    
    # Process each document that passed filters
    for doc_path in tqdm(doc_files, desc="Chunking documents"):
        # Read document text
        with open(doc_path, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        
        # Extract metadata from path structure
        metadata = extract_path_metadata(str(doc_path), corpus_dir)
        
        # Split document into chunks
        doc_chunks = chunk_by_sentences(text, str(doc_path))
        
        # Store each chunk with its metadata
        for chunk_uid, raw_text, text_for_scoring, sentence_count in doc_chunks:
            all_chunks.append({
                'file_path': str(doc_path),
                'chunk_uid': chunk_uid,
                'raw_text': raw_text,
                'text_for_scoring': text_for_scoring,
                'sentence_count': sentence_count,
                # Metadata from folder structure (changed category to doc_type)
                'doc_type': metadata['doc_type'],  # Changed from 'category'
                'year': metadata['year'],
                'document_folder': metadata['document_folder'],
                'filename': metadata['filename']
            })
    
    # ===== CREATE DATAFRAME AND SAVE =====
    chunks_df = pd.DataFrame(all_chunks)
    fs.save_data(chunks_df, "chunked_corpus", "Other_data", "csv")
    
    # ===== DISPLAY FINAL STATISTICS =====
    print(f"\n{'='*60}")
    print("CHUNKING COMPLETE")
    print(f"{'='*60}")
    print(f"\n✓ Processed {len(chunks_df)} chunks from {chunks_df['file_path'].nunique()} documents")
    print(f"  Avg sentences/chunk: {chunks_df['sentence_count'].mean():.1f}")
    print(f"  Empty scoring text: {(chunks_df['text_for_scoring'] == '').sum()}")
    
    # Show metadata distribution in chunks
    if 'doc_type' in chunks_df.columns:  # Changed from 'category'
        print(f"\n📊 Metadata distribution in chunks:")
        print(f"  Document types: {chunks_df['doc_type'].nunique()} unique")
        print(f"  Years: {chunks_df['year'].nunique()} unique")
        print(f"  Document folders: {chunks_df['document_folder'].nunique()} unique")
        
        # Show top doc_types by chunk count (changed from categories)
        if chunks_df['doc_type'].notna().any():
            print(f"\n  Top document types by chunk count:")
            for doc_type, count in chunks_df['doc_type'].value_counts().head(5).items():
                print(f"    - {doc_type}: {count} chunks")
        
        # Show year distribution in chunks
        if chunks_df['year'].notna().any():
            print(f"\n  Year distribution:")
            for year, count in chunks_df['year'].value_counts().sort_index().items():
                print(f"    - {year}: {count} chunks")
    
    # Save checkpoint
    fs.save_config("checkpoint1_chunks")


PROCESSING CORPUS (WITH FILTERING)

Found 1582 total documents in nested folders

✅ 1582 documents passed filters

CHUNKING DOCUMENTS



Chunking documents: 100%|██████████| 1582/1582 [00:01<00:00, 1180.50it/s]


✓ Saved: Other_data/chunked_corpus.csv

CHUNKING COMPLETE

✓ Processed 2900 chunks from 1378 documents
  Avg sentences/chunk: 8.5
  Empty scoring text: 59

📊 Metadata distribution in chunks:
  Document types: 0 unique
  Years: 0 unique
  Document folders: 0 unique
✓ Config saved: config_checkpoint1_chunks_20251030_170922.json


✅ **CHECKPOINT 1 COMPLETE** - Corpus chunked and saved

**Resume**: Load `chunks_df` from `Other_data/chunked_corpus.csv`

In [100]:
# ============================================================
# CELL 2.1: TOKENIZATION FOR VOCAB BUILDING
# ============================================================

_tok_re = re.compile(CONFIG["tokenize"]["pattern"])

def tokenize(text: str) -> list:
    if CONFIG["tokenize"]["lower"]:
        text = text.lower()
    toks = _tok_re.findall(text)
    keep = []
    mn = CONFIG["tokenize"]["min_len"]
    mx = CONFIG["tokenize"]["max_len"]
    for t in toks:
        if not CONFIG["tokenize"]["keep_hyphen"]:
            t = t.replace("-", "")
        if mn <= len(t) <= mx:
            keep.append(t)
    return keep

def read_text(path: Path) -> str:
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return path.read_text(encoding=enc, errors="ignore")
        except Exception:
            pass
    return path.read_text(errors="ignore")

print("✓ Tokenizer ready")

✓ Tokenizer ready


In [101]:
# ============================================================
# BUILD VOCABULARY FROM CHUNKED CORPUS (DIRECT PATH)
# ============================================================

print(f"\n{'='*60}")
print("BUILDING VOCABULARY FROM CHUNKED CORPUS")
print(f"{'='*60}")

# Try to construct the path directly from CONFIG
workflow_base = Path(CONFIG["paths"]["workflow_base"])
workflow_name = f"{CONFIG['workflow']['model_type']}-{CONFIG['workflow']['topic']}"

# Find the most recent workflow directory (or use a specific version)
if CONFIG['workflow']['version']:
    workflow_dir = workflow_base / f"{workflow_name}_{CONFIG['workflow']['version']}"
else:
    # Find the most recent version
    matching_dirs = list(workflow_base.glob(f"{workflow_name}_*"))
    if matching_dirs:
        workflow_dir = sorted(matching_dirs)[-1]  # Get the most recent
    else:
        print(f"❌ Error: No workflow directory found matching {workflow_name}")
        workflow_dir = None

if workflow_dir and workflow_dir.exists():
    chunked_corpus_path = workflow_dir / "Other_data" / "chunked_corpus.csv"
    print(f"📂 Using workflow directory: {workflow_dir}")
else:
    # Fallback: look in current working directory
    chunked_corpus_path = Path("Other_data") / "chunked_corpus.csv"
    print(f"📂 Using relative path: {chunked_corpus_path}")

if not chunked_corpus_path.exists():
    print(f"❌ Error: Chunked corpus not found at {chunked_corpus_path}")
    print("   Please run the chunking step first.")
    print(f"\n   Looking for files in: {chunked_corpus_path.parent}")
    if chunked_corpus_path.parent.exists():
        files = list(chunked_corpus_path.parent.glob("*.csv"))
        print(f"   Found {len(files)} CSV files:")
        for f in files[:10]:
            print(f"     - {f.name}")
else:
    # Load the chunks DataFrame
    print(f"✓ Loading from: {chunked_corpus_path}")
    chunks_df = pd.read_csv(chunked_corpus_path)
    
    print(f"\n✓ Loaded {len(chunks_df)} chunks from {chunks_df['file_path'].nunique()} documents")
    
    # Show filtering info if available
    if 'category' in chunks_df.columns and chunks_df['category'].notna().any():
        print(f"  Categories: {chunks_df['category'].nunique()} unique")
        print(f"  Years: {chunks_df['year'].nunique()} unique")
    
    # Build vocabulary from the processed text
    print("\n📝 Tokenizing chunks...")
    
    term_freq = Counter()
    doc_freq = Counter()
    chunk_tokens = []
    
    for idx, row in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Processing chunks"):
        # Use the already-processed text_for_scoring
        text = row['text_for_scoring']
        
        # Skip empty texts
        if pd.isna(text) or text.strip() == '':
            continue
        
        # Tokenize the text
        toks = tokenize(text)
        
        # Store tokens with chunk ID for reference
        chunk_tokens.append({
            'chunk_uid': row['chunk_uid'],
            'tokens': toks
        })
        
        # Update frequencies
        term_freq.update(toks)
        doc_freq.update(set(toks))  # Count each term once per chunk
    
    print(f"\n✓ Processed {len(chunk_tokens)} chunks with text")
    print(f"  Total tokens: {sum(term_freq.values()):,}")
    print(f"  Unique terms: {len(term_freq):,}")
    
    # Filter vocabulary based on minimum document frequency
    min_df = CONFIG["vocab"]["min_df"]
    max_vocab = CONFIG["vocab"]["max_vocab"]
    
    print(f"\n🔍 Filtering vocabulary...")
    print(f"  Min document frequency: {min_df}")
    print(f"  Max vocabulary size: {max_vocab}")
    
    # Get terms that appear in at least min_df chunks
    vocab_candidates = [
        (term, freq) for term, freq in term_freq.items()
        if doc_freq[term] >= min_df
    ]
    
    # Sort by frequency and take top max_vocab terms
    vocab_candidates.sort(key=lambda x: x[1], reverse=True)
    vocab_candidates = vocab_candidates[:max_vocab]
    terms = [term for term, _ in vocab_candidates]
    
    print(f"\n✓ Filtered vocabulary: {len(terms)} terms")
    
    # Show some statistics
    if len(terms) > 0:
        print(f"\n📊 Vocabulary statistics:")
        print(f"  Most common terms:")
        for term, freq in vocab_candidates[:10]:
            print(f"    - '{term}': {freq:,} occurrences (in {doc_freq[term]} chunks)")
        
        # Show filtering impact
        removed = len(term_freq) - len(terms)
        print(f"\n  Removed {removed:,} rare terms (appeared in < {min_df} chunks)")
    
    # Save vocabulary
    vocab_df = pd.DataFrame({
        'term': terms,
        'term_freq': [term_freq[t] for t in terms],
        'doc_freq': [doc_freq[t] for t in terms]
    })
    fs.save_data(vocab_df, "vocabulary", "Other_data", "csv")
    
    # Save frequencies for all terms (not just vocabulary)
    freq_data = {
        'term_freq': dict(term_freq),
        'doc_freq': dict(doc_freq),
        'n_chunks': len(chunk_tokens),
        'n_documents': chunks_df['file_path'].nunique()
    }
    fs.save_data(freq_data, "term_frequencies", "Other_data", "json")
    
    # Save the tokenized chunks for later use
    tokens_df = pd.DataFrame(chunk_tokens)
    fs.save_data(tokens_df, "chunk_tokens", "Other_data", "csv")
    
    fs.save_config("checkpoint2_vocab")
    
    print(f"\n{'='*60}")
    print("VOCABULARY BUILDING COMPLETE")
    print(f"{'='*60}")


BUILDING VOCABULARY FROM CHUNKED CORPUS
📂 Using workflow directory: workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1
✓ Loading from: workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1\Other_data\chunked_corpus.csv

✓ Loaded 2900 chunks from 1378 documents

📝 Tokenizing chunks...


Processing chunks: 100%|██████████| 2900/2900 [00:00<00:00, 8761.04it/s]



✓ Processed 2841 chunks with text
  Total tokens: 236,266
  Unique terms: 30,762

🔍 Filtering vocabulary...
  Min document frequency: 5
  Max vocabulary size: 50000

✓ Filtered vocabulary: 6661 terms

📊 Vocabulary statistics:
  Most common terms:
    - 'slavernij': 2,180 occurrences (in 935 chunks)
    - 'nederlandse': 1,248 occurrences (in 792 chunks)
    - 'racisme': 1,036 occurrences (in 448 chunks)
    - 'koloniale': 1,004 occurrences (in 535 chunks)
    - 'nederland': 970 occurrences (in 600 chunks)
    - 'mensen': 901 occurrences (in 569 chunks)
    - 'den': 838 occurrences (in 514 chunks)
    - 'utrecht': 799 occurrences (in 357 chunks)
    - 'onderzoek': 790 occurrences (in 509 chunks)
    - 'amsterdam': 769 occurrences (in 465 chunks)

  Removed 24,101 rare terms (appeared in < 5 chunks)
✓ Saved: Other_data/vocabulary.csv
✓ Saved: Other_data/term_frequencies.json
✓ Saved: Other_data/chunk_tokens.csv
✓ Config saved: config_checkpoint2_vocab_20251030_171524.json

VOCABULARY BUI

✅ **CHECKPOINT 2 COMPLETE** - Vocabulary built and saved

**Resume**: Load `terms`, `term_freq`, `doc_freq` from saved files

---
# CHECKPOINT 3: Dictionary Expansion
---

Expand dictionary keywords using SBERT semantic similarity to find related terms.

⚠️ **MANUAL CURATION REQUIRED**: Review and curate the suggested expansions before proceeding to Checkpoint 4.


In [ ]:
# ============================================================
# CELL 3.1: DICTIONARY EXPANSION USING SBERT
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 3 START - DICTIONARY EXPANSION")
print(f"{'='*60}")

# =====================
# 1) LOAD REQUIRED DATA
# =====================

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    fs = WorkflowFileSystem(CONFIG)
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    workflow_name = f"{CONFIG['workflow']['model_type']}-{CONFIG['workflow']['topic']}"
    if CONFIG['workflow']['version']:
        workflow_dir = workflow_base / f"{workflow_name}_{CONFIG['workflow']['version']}"
    else:
        matching_dirs = list(workflow_base.glob(f"{workflow_name}_*"))
        workflow_dir = max(matching_dirs, key=lambda p: p.stat().st_mtime) if matching_dirs else None
    if workflow_dir and workflow_dir.exists():
        fs = WorkflowFileSystem.from_existing(str(workflow_dir), CONFIG)
    print(f"✓ Workflow directory: {fs.workflow_dir}")

# Verify SBERT model is loaded
if 'st_model' not in globals():
    from sentence_transformers import SentenceTransformer
    if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
        model_path = CONFIG['paths']['pretrained_model_path']
        if Path(model_path).exists():
            st_model = SentenceTransformer(model_path)
            print(f"✓ Loaded pretrained SBERT model from: {model_path}")
        else:
            st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
            print(f"⚠ Pretrained path not found, using base model: {CONFIG['model']['base_model_name']}")
    else:
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"✓ Loaded base SBERT model: {CONFIG['model']['base_model_name']}")

# Load vocabulary
vocab_path = fs.folders['Other_data'] / 'vocabulary.json'
if not vocab_path.exists():
    raise FileNotFoundError(
        f"❌ Vocabulary not found at {vocab_path}\n"
        f"   Please run Checkpoint 2 first to build the vocabulary!"
    )

with open(vocab_path, 'r', encoding='utf-8') as f:
    vocab_data = json.load(f)
    
terms = vocab_data['terms']
print(f"✓ Loaded vocabulary: {len(terms)} terms")

# Load dictionary
dict_file = Path(CONFIG['paths']['dictionary_path'])
if not dict_file.exists():
    raise FileNotFoundError(f"❌ Dictionary not found at {dict_file}")

with open(dict_file, 'r', encoding='utf-8') as f:
    dictionary = json.load(f)

print(f"✓ Loaded dictionary: {len(dictionary)} topics")
for topic, keywords in dictionary.items():
    print(f"  - {topic}: {len(keywords)} keywords")

# =====================
# 2) ENCODE VOCABULARY
# =====================

print(f"\n{'='*60}")
print("ENCODING VOCABULARY WITH SBERT")
print(f"{'='*60}")

# Encode all vocabulary terms
print(f"Encoding {len(terms)} vocabulary terms...")
vocab_embeddings = st_model.encode(terms, show_progress_bar=True, convert_to_numpy=True)
print(f"✓ Vocabulary embeddings shape: {vocab_embeddings.shape}")

# =====================
# 3) EXPAND DICTIONARY
# =====================

print(f"\n{'='*60}")
print("EXPANDING DICTIONARY WITH SEMANTIC SIMILARITY")
print(f"{'='*60}")

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Configuration for expansion
TOP_N_SUGGESTIONS = CONFIG.get('dictionary_expansion', {}).get('top_n_suggestions', 20)
MIN_SIMILARITY = CONFIG.get('dictionary_expansion', {}).get('min_similarity', 0.5)

# Create suggestions directory
suggestions_dir = fs.folders['Dictionary'] / 'Dictionary_suggestions'
suggestions_dir.mkdir(exist_ok=True)
print(f"✓ Suggestions directory: {suggestions_dir}")

# Process each topic
all_suggestions = {}

for topic, keywords in dictionary.items():
    print(f"\n{'='*60}")
    print(f"Processing topic: {topic}")
    print(f"{'='*60}")
    print(f"Keywords: {keywords}")
    
    # Encode topic keywords
    keyword_embeddings = st_model.encode(keywords, show_progress_bar=False, convert_to_numpy=True)
    
    # Calculate average topic vector (centroid)
    topic_centroid = np.mean(keyword_embeddings, axis=0, keepdims=True)
    
    # Calculate similarity with all vocabulary terms
    similarities = cosine_similarity(topic_centroid, vocab_embeddings)[0]
    
    # Get top N similar terms (excluding exact matches)
    term_scores = list(zip(terms, similarities))
    term_scores = sorted(term_scores, key=lambda x: x[1], reverse=True)
    
    # Filter out existing keywords and low similarity
    suggestions = []
    for term, score in term_scores:
        if term.lower() not in [k.lower() for k in keywords]:
            if score >= MIN_SIMILARITY:
                suggestions.append({'term': term, 'similarity': float(score)})
        if len(suggestions) >= TOP_N_SUGGESTIONS:
            break
    
    all_suggestions[topic] = suggestions
    
    # Save to CSV for manual curation
    csv_path = suggestions_dir / f"{topic.lower().replace(' ', '_')}_suggestions.csv"
    df_suggestions = pd.DataFrame(suggestions)
    df_suggestions.to_csv(csv_path, index=False)
    
    print(f"✓ Found {len(suggestions)} suggestions (similarity >= {MIN_SIMILARITY:.2f})")
    print(f"  Top 5: {[s['term'] for s in suggestions[:5]]}")
    print(f"✓ Saved to: {csv_path}")

# =====================
# 4) SAVE SUMMARY
# =====================

summary_path = suggestions_dir / '_expansion_summary.json'
summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'config': {
        'top_n_suggestions': TOP_N_SUGGESTIONS,
        'min_similarity': MIN_SIMILARITY,
        'model': CONFIG['model']['base_model_name']
    },
    'topics': {
        topic: {
            'original_keywords': len(keywords),
            'suggestions_generated': len(all_suggestions[topic])
        }
        for topic, keywords in dictionary.items()
    }
}

with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*60}")
print("DICTIONARY EXPANSION COMPLETE")
print(f"{'='*60}")
print(f"✓ Suggestions saved to: {suggestions_dir}")
print(f"✓ Summary saved to: {summary_path}")
print(f"\n⚠️  NEXT STEP: Manually review and curate the suggestions")
print(f"   Then update your dictionary file with approved expansions")
print(f"   After curation, proceed to Checkpoint 4")


✅ **CHECKPOINT 3 COMPLETE** - Dictionary expansion suggestions generated

**Resume**: Review suggestions in `Dictionary/Dictionary_suggestions/`, update dictionary with approved terms

---
# CHECKPOINT 4: Build Topic Vectors
---

Create weighted topic vectors from the curated/expanded dictionary using SBERT embeddings.

These topic vectors will be used in Checkpoint 5 for scoring chunks.

In [ ]:
# ============================================================
# CELL 4.1: BUILD TOPIC VECTORS
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 4 START - BUILDING TOPIC VECTORS")
print(f"{'='*60}")

# =====================
# 1) LOAD CURATED DICTIONARY
# =====================

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    fs = WorkflowFileSystem(CONFIG)
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    workflow_name = f"{CONFIG['workflow']['model_type']}-{CONFIG['workflow']['topic']}"
    if CONFIG['workflow']['version']:
        workflow_dir = workflow_base / f"{workflow_name}_{CONFIG['workflow']['version']}"
    else:
        matching_dirs = list(workflow_base.glob(f"{workflow_name}_*"))
        workflow_dir = max(matching_dirs, key=lambda p: p.stat().st_mtime) if matching_dirs else None
    if workflow_dir and workflow_dir.exists():
        fs = WorkflowFileSystem.from_existing(str(workflow_dir), CONFIG)
    print(f"✓ Workflow directory: {fs.workflow_dir}")

# Verify SBERT model is loaded
if 'st_model' not in globals():
    from sentence_transformers import SentenceTransformer
    if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
        model_path = CONFIG['paths']['pretrained_model_path']
        if Path(model_path).exists():
            st_model = SentenceTransformer(model_path)
            print(f"✓ Loaded pretrained SBERT model from: {model_path}")
        else:
            st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
            print(f"⚠ Pretrained path not found, using base model: {CONFIG['model']['base_model_name']}")
    else:
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"✓ Loaded base SBERT model: {CONFIG['model']['base_model_name']}")

# Load dictionary (should be curated/expanded by now)
dict_file = Path(CONFIG['paths']['dictionary_path'])
if not dict_file.exists():
    raise FileNotFoundError(f"❌ Dictionary not found at {dict_file}")

with open(dict_file, 'r', encoding='utf-8') as f:
    dictionary = json.load(f)

print(f"✓ Loaded dictionary: {len(dictionary)} topics")
for topic, keywords in dictionary.items():
    print(f"  - {topic}: {len(keywords)} keywords")

# =====================
# 2) BUILD TOPIC VECTORS
# =====================

print(f"\n{'='*60}")
print("BUILDING TOPIC VECTORS FROM KEYWORDS")
print(f"{'='*60}")

import numpy as np

topic_vectors = {}
topic_metadata = {}

for topic, keywords in dictionary.items():
    print(f"\nProcessing topic: {topic}")
    print(f"  Keywords: {keywords}")
    
    if not keywords:
        print(f"  ⚠️  WARNING: No keywords for topic '{topic}', skipping")
        continue
    
    # Encode all keywords for this topic
    keyword_embeddings = st_model.encode(keywords, show_progress_bar=False, convert_to_numpy=True)
    
    # Create topic vector as the mean of keyword embeddings
    topic_vector = np.mean(keyword_embeddings, axis=0)
    
    # Normalize the topic vector (L2 normalization for cosine similarity)
    topic_vector = topic_vector / np.linalg.norm(topic_vector)
    
    # Store
    topic_vectors[topic] = topic_vector
    topic_metadata[topic] = {
        'num_keywords': len(keywords),
        'keywords': keywords,
        'vector_shape': topic_vector.shape,
        'vector_norm': float(np.linalg.norm(topic_vector))
    }
    
    print(f"  ✓ Created vector of shape {topic_vector.shape}")
    print(f"  ✓ Vector norm: {np.linalg.norm(topic_vector):.4f}")

# =====================
# 3) SAVE TOPIC VECTORS
# =====================

print(f"\n{'='*60}")
print("SAVING TOPIC VECTORS")
print(f"{'='*60}")

# Save topic vectors as .npy file
topic_vec_path = fs.folders['Other_data'] / 'topic_vectors.npy'
np.save(topic_vec_path, topic_vectors)
print(f"✓ Saved topic vectors to: {topic_vec_path}")

# Save metadata as JSON
topic_meta_path = fs.folders['Other_data'] / 'topic_vectors_meta.json'
# Convert numpy types to native Python types for JSON serialization
metadata_serializable = {}
for topic, meta in topic_metadata.items():
    metadata_serializable[topic] = {
        'num_keywords': meta['num_keywords'],
        'keywords': meta['keywords'],
        'vector_shape': list(meta['vector_shape']),
        'vector_norm': meta['vector_norm']
    }

with open(topic_meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata_serializable, f, indent=2)
print(f"✓ Saved topic metadata to: {topic_meta_path}")

# =====================
# 4) VERIFY & SUMMARY
# =====================

print(f"\n{'='*60}")
print("TOPIC VECTORS SUMMARY")
print(f"{'='*60}")
print(f"Total topics: {len(topic_vectors)}")
for topic, vec in topic_vectors.items():
    print(f"  - {topic}: vector shape {vec.shape}, {topic_metadata[topic]['num_keywords']} keywords")

print(f"\n✓ Topic vectors ready for Checkpoint 5 (chunk scoring)")
print(f"  Files created:")
print(f"    - {topic_vec_path}")
print(f"    - {topic_meta_path}")


✅ **CHECKPOINT 4 COMPLETE** - Topic vectors built and saved

**Resume**: Load topic vectors from `Other_data/topic_vectors.npy` and metadata from `Other_data/topic_vectors_meta.json`

In [ ]:
---
# CHECKPOINT 5: Chunk Scoring & Confidence Classification
---

Score all chunks and classify by confidence level (High/Low/None).
# ============================================================
# CELL 5.1: SCORE CHUNKS
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 5 START - LOADING REQUIRED DATA")
print(f"{'='*60}")

# =====================
# VALIDATE REQUIRED FILES
# =====================

# Initialize/verify fs object first (needed for path construction)
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['model_type']
    topic = CONFIG['workflow']['topic']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# CRITICAL VALIDATION: Check if topic vectors exist
topic_vec_path = fs.folders['Other_data'] / 'topic_vectors.npy'
topic_meta_path = fs.folders['Other_data'] / 'topic_vectors_meta.json'

if not topic_vec_path.exists():
    raise FileNotFoundError(
        f"❌ Topic vectors not found at {topic_vec_path}\n"
        f"   Please run Checkpoint 4 to generate topic vectors first!"
    )
if not topic_meta_path.exists():
    raise FileNotFoundError(
        f"❌ Topic metadata not found at {topic_meta_path}\n"
        f"   Please run Checkpoint 4 to generate topic metadata first!"
    )

print(f"✓ Validated required files:")
print(f"  - Topic vectors: {topic_vec_path}")
print(f"  - Topic metadata: {topic_meta_path}")

# =====================
# LOAD DATA
# =====================

# Load chunks DataFrame
chunks_path = fs.folders['Other_data'] / 'chunked_corpus.csv'
if not chunks_path.exists():
    raise FileNotFoundError(f"Chunked corpus not found at {chunks_path}")

chunks_df = pd.read_csv(chunks_path)
print(f"✓ Loaded chunks: {len(chunks_df)} chunks")

# Load topic vectors
topic2vec = np.load(topic_vec_path, allow_pickle=True).item()
with open(topic_meta_path, 'r') as f:
    topic_meta = json.load(f)

print(f"✓ Loaded topic vectors: {len(topic2vec)} topics")
print(f"  Topics: {', '.join(topic2vec.keys())}")

# =====================
# SCORE CHUNKS
# =====================

print(f"\n{'='*60}")
print("SCORING CHUNKS")
print(f"{'='*60}")

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

# Score all chunks
records = []
for idx, chunk in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Scoring"):
    text = chunk['text_for_scoring']
    
    # Check if text is actually a valid string (not NaN, not empty, not a float)
    if isinstance(text, str) and text.strip():
        dv = st_embed([text])[0]
        row = {
            'filename': chunk['file_path'],
            'chunk_id': chunk['chunk_uid'],
            'sentence_count': chunk['sentence_count'],
            'raw_text': chunk['raw_text'],
            'text_for_scoring': text,
        }
        for topic, tv in topic2vec.items():
            row[f'cos_{topic}'] = cosine(dv, tv)
    else:
        # Handle invalid text (NaN, empty, or float values)
        row = {
            'filename': chunk['file_path'],
            'chunk_id': chunk['chunk_uid'],
            'sentence_count': chunk['sentence_count'],
            'raw_text': chunk['raw_text'],
            'text_for_scoring': '',
        }
        for topic in topic2vec.keys():
            row[f'cos_{topic}'] = 0.0
    
    records.append(row)

all_scores_df = pd.DataFrame(records)
print(f"\n✓ Scored {len(all_scores_df)} chunks across {len(topic2vec)} topics")

# Calculate metrics
topic_cols = [col for col in all_scores_df.columns if col.startswith('cos_')]
all_scores_df['max_score'] = all_scores_df[topic_cols].max(axis=1)
all_scores_df['primary_topic'] = all_scores_df[topic_cols].idxmax(axis=1).str.replace('cos_', '')

topic_scores = all_scores_df[topic_cols].values
sorted_scores = np.sort(topic_scores, axis=1)[:, ::-1]
all_scores_df['score_margin'] = sorted_scores[:, 0] - sorted_scores[:, 1]

print(f"✓ Calculated max_score, primary_topic, and score_margin")

# ============================================================
# CELL 5.2: CONFIDENCE CLASSIFICATION
# ============================================================

print(f"\n{'='*60}")
print("CONFIDENCE CLASSIFICATION")
print(f"{'='*60}")

# Thresholds
HIGH_SCORE = CONFIG['scoring']['high_confidence_score']
HIGH_MARGIN = CONFIG['scoring']['high_confidence_margin']
LOW_SCORE = CONFIG['scoring']['low_confidence_score']
LOW_MARGIN = CONFIG['scoring']['low_confidence_margin']

# Classify
high_mask = (
    (all_scores_df['max_score'] >= HIGH_SCORE) & 
    (all_scores_df['score_margin'] >= HIGH_MARGIN)
)

low_mask = (
    (all_scores_df['max_score'] >= LOW_SCORE) & 
    (all_scores_df['score_margin'] >= LOW_MARGIN) &
    ~high_mask
)

no_mask = ~(high_mask | low_mask)

high_df = all_scores_df[high_mask].copy()
low_df = all_scores_df[low_mask].copy()
no_df = all_scores_df[no_mask].copy()

high_df['confidence_level'] = 'high'
low_df['confidence_level'] = 'low'
no_df['confidence_level'] = 'none'

# Save
fs.save_data(high_df, "scores_high_confidence", "Cosine_labeling", "csv")
fs.save_data(low_df, "scores_low_confidence", "Cosine_labeling", "csv")
fs.save_data(no_df, "scores_no_confidence", "Cosine_labeling", "csv")

all_labeled = pd.concat([high_df, low_df, no_df], ignore_index=True)
fs.save_data(all_labeled, "scores_all_labeled", "Cosine_labeling", "csv")

# Report
total = len(all_scores_df)
print(f"\nTotal chunks: {total}")
print(f"\n1. HIGH CONFIDENCE: {len(high_df)} ({len(high_df)/total*100:.1f}%)")
print(f"   Mean score: {high_df['max_score'].mean():.3f}, Mean margin: {high_df['score_margin'].mean():.3f}")
print(f"\n2. LOW CONFIDENCE: {len(low_df)} ({len(low_df)/total*100:.1f}%)")
print(f"   Mean score: {low_df['max_score'].mean():.3f}, Mean margin: {low_df['score_margin'].mean():.3f}")
print(f"\n3. NO CONFIDENCE: {len(no_df)} ({len(no_df)/total*100:.1f}%)")
print(f"   Mean score: {no_df['max_score'].mean():.3f}, Mean margin: {no_df['score_margin'].mean():.3f}")

fs.save_config("checkpoint5_scoring")


---
# CHECKPOINT 6: Training Data Preparation
---

Create train/val splits from confidence tiers.

In [6]:
# ============================================================
# CELL 6: LOAD LABELED SCORES
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 6 START - LOADING LABELED SCORE FILES")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['model_type']
    topic = CONFIG['workflow']['topic']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load the three CSV files using fs.folders
high_path = fs.folders['Cosine_labeling'] / 'scores_high_confidence.csv'
low_path = fs.folders['Cosine_labeling'] / 'scores_low_confidence.csv'
no_path = fs.folders['Cosine_labeling'] / 'scores_no_confidence.csv'

if not high_path.exists() or not low_path.exists() or not no_path.exists():
    print(f"❌ Error: One or more score files not found")
    print(f"   Looking in: {fs.folders['Cosine_labeling']}")
    print(f"   - {high_path.name} {'✓' if high_path.exists() else '✗'}")
    print(f"   - {low_path.name} {'✓' if low_path.exists() else '✗'}")
    print(f"   - {no_path.name} {'✓' if no_path.exists() else '✗'}")
    raise FileNotFoundError("Required score files not found. Please run CHECKPOINT 5 first.")
else:
    high_df = pd.read_csv(high_path)
    low_df = pd.read_csv(low_path)
    no_df = pd.read_csv(no_path)
    
    print(f"✓ Loaded score files from: {fs.folders['Cosine_labeling']}")
    print(f"  High confidence: {len(high_df)} chunks")
    print(f"  Low confidence:  {len(low_df)} chunks")
    print(f"  No confidence:   {len(no_df)} chunks")
    print(f"  Total:           {len(high_df) + len(low_df) + len(no_df)} chunks")


CHECKPOINT 6 START - LOADING LABELED SCORE FILES
✓ Loaded score files from: C:\Users\Home\policy-analysis\workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1\Cosine_labeling
  High confidence: 69 chunks
  Low confidence:  719 chunks
  No confidence:   2112 chunks
  Total:           2900 chunks


In [ ]:
# ============================================================
# CELL 6.1: PREPARE LABELED DATA (with cosine scores for multi-label)
# ============================================================

print(f"\n{'='*60}")
print("PREPARING LABELED DATA")
print(f"{'='*60}")

# High confidence = labeled data
df_labeled = high_df.copy()
df_labeled['text'] = df_labeled['raw_text']
df_labeled['label'] = df_labeled['primary_topic']

# Create label mapping
label2id = {label: idx for idx, label in enumerate(sorted(df_labeled['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}
df_labeled['label_id'] = df_labeled['label'].map(label2id)
df_labeled['is_pseudo'] = False

print(f"\nLabel mapping:")
for label, idx in label2id.items():
    count = (df_labeled['label'] == label).sum()
    print(f"  {idx}: {label} ({count} examples)")

print(f"\nTotal labeled examples: {len(df_labeled)}")

# =====================
# CRITICAL: VALIDATE COSINE SCORE COLUMNS
# =====================

# Check if we have cosine score columns
cosine_cols = [c for c in df_labeled.columns if c.startswith('cos_')]

if not cosine_cols:
    raise ValueError(
        f"❌ No cosine score columns found in labeled data!\n"
        f"   Expected columns like: cos_Topic1, cos_Topic2, etc.\n"
        f"   Please ensure Checkpoint 5 completed successfully and saved cosine scores.\n"
        f"   Available columns: {list(df_labeled.columns)}"
    )

if len(cosine_cols) != len(label2id):
    print(f"⚠️  WARNING: Found {len(cosine_cols)} cosine columns but {len(label2id)} topics")
    print(f"   Cosine columns: {cosine_cols}")
    print(f"   Topics: {list(label2id.keys())}")
    
print(f"\n✓ Validated {len(cosine_cols)} cosine score columns for multi-label support")
print(f"  Topics: {list(label2id.keys())}")
print(f"  Cosine columns: {cosine_cols}")


In [26]:
# ============================================================
# CELL 6.2: PREPARE PSEUDO-LABELED & UNLABELED DATA
# ============================================================

# Use existing CONFIG or create defaults
if 'CONFIG' not in globals():
    CONFIG = {}
if 'sampling' not in CONFIG:
    CONFIG['sampling'] = {
        "unlabeled_multiplier": 10,  # Max unlabeled = labeled_size * 3
        "pseudo_multiplier": 10      # Max pseudo = labeled_size * 10
    }

print(f"\n{'='*60}")
print("PREPARING PSEUDO-LABELED & UNLABELED DATA")
print(f"{'='*60}")

# =====================
# PREPARE PSEUDO-LABELED DATA
# =====================

# Pseudo-labeled pool (low confidence predictions)
df_pseudo = low_df.copy()
df_pseudo['text'] = df_pseudo['raw_text']
df_pseudo['label'] = df_pseudo['primary_topic']
df_pseudo['label_id'] = df_pseudo['label'].map(label2id)
df_pseudo['is_pseudo'] = True

print(f"\nPseudo-labeled pool: {len(df_pseudo)} chunks")

# Sample pseudo-labeled data for balance
max_pseudo = len(df_labeled) * CONFIG["sampling"]["pseudo_multiplier"]
pseudo_cols = ['text', 'label', 'label_id', 'is_pseudo'] + [c for c in df_pseudo.columns if c.startswith('cos_')]
pseudo_cols = [c for c in pseudo_cols if c in df_pseudo.columns]

if len(df_pseudo) > max_pseudo:
    df_pseudo_sampled = df_pseudo[pseudo_cols].sample(n=max_pseudo, random_state=42)
    print(f"  Sampled: {len(df_pseudo_sampled)} (to maintain balance)")
else:
    df_pseudo_sampled = df_pseudo[pseudo_cols].copy()
    print(f"  Using all: {len(df_pseudo_sampled)}")

# =====================
# PREPARE UNLABELED DATA
# =====================

# Unlabeled pool (no confidence predictions)
df_unlabeled = no_df[['raw_text']].copy()
df_unlabeled.rename(columns={'raw_text': 'text'}, inplace=True)
df_unlabeled['label'] = 'UNLABELED'
df_unlabeled['label_id'] = -1
df_unlabeled['is_pseudo'] = False

# Clean: remove empty/null text
df_unlabeled = df_unlabeled[df_unlabeled['text'].notna()].copy()
df_unlabeled = df_unlabeled[df_unlabeled['text'].astype(str).str.strip() != ''].copy()

print(f"\nUnlabeled pool: {len(df_unlabeled)} chunks")

# Sample unlabeled data for balance
max_unlabeled = len(df_labeled) * CONFIG["sampling"]["unlabeled_multiplier"]
if len(df_unlabeled) > max_unlabeled:
    df_unlabeled_sampled = df_unlabeled.sample(n=max_unlabeled, random_state=42)
    print(f"  Sampled: {len(df_unlabeled_sampled)} (to maintain balance)")
else:
    df_unlabeled_sampled = df_unlabeled.copy()
    print(f"  Using all: {len(df_unlabeled_sampled)}")

# =====================
# SUMMARY
# =====================

print(f"\n{'='*60}")
print("DATA PREPARATION SUMMARY")
print(f"{'='*60}")
print(f"  Labeled:     {len(df_labeled)}")
print(f"  Pseudo:      {len(df_pseudo_sampled)}")
print(f"  Unlabeled:   {len(df_unlabeled_sampled)}")
print(f"  Total pool:  {len(df_labeled) + len(df_pseudo_sampled) + len(df_unlabeled_sampled)}")


PREPARING PSEUDO-LABELED & UNLABELED DATA

Pseudo-labeled pool: 719 chunks
  Sampled: 690 (to maintain balance)

Unlabeled pool: 2112 chunks
  Sampled: 207 (to maintain balance)

DATA PREPARATION SUMMARY
  Labeled:     69
  Pseudo:      690
  Unlabeled:   207
  Total pool:  966


In [27]:
# ============================================================
# CELL 6.3: CREATE DATASET OPTIONS & TRAIN/VAL SPLIT
# (Same as v6 - keeping existing workflow)
# ============================================================

from sklearn.model_selection import train_test_split

print(f"\n{'='*60}")
print("CREATING DATASET OPTIONS & TRAIN/VAL SPLIT")
print(f"{'='*60}")

# =====================
# STEP 1: GROUP DATA INTO OPTIONS FIRST
# =====================

print(f"\nStep 1: Grouping data into options...")

# =====================
# OPTION 1: LABELED ONLY
# =====================
data_opt1 = df_labeled.copy()

# =====================
# OPTION 2: LABELED + PSEUDO-LABELED
# =====================
data_opt2 = pd.concat([df_labeled, df_pseudo_sampled], ignore_index=True)

# =====================
# OPTION 3: LABELED + UNLABELED
# =====================
data_opt3 = pd.concat([df_labeled, df_unlabeled_sampled], ignore_index=True)

# =====================
# OPTION 4: ALL (LABELED + PSEUDO + UNLABELED)
# =====================
data_opt4 = pd.concat([df_labeled, df_pseudo_sampled, df_unlabeled_sampled], ignore_index=True)

print(f"  Option 1 (Labeled only):          {len(data_opt1):>6} examples")
print(f"  Option 2 (Labeled + Pseudo):      {len(data_opt2):>6} examples")
print(f"  Option 3 (Labeled + Unlabeled):   {len(data_opt3):>6} examples")
print(f"  Option 4 (All) ⭐ RECOMMENDED:    {len(data_opt4):>6} examples")

# =====================
# STEP 2: SPLIT EACH OPTION INTO TRAIN/VAL
# =====================

print(f"\nStep 2: Splitting each option into train/val...")

def split_with_stratification(data, option_name):
    """Split data into train/val, using stratification if possible."""
    labeled_data = data[data['label'] != 'UNLABELED'].copy()
    unlabeled_data = data[data['label'] == 'UNLABELED'].copy()
    
    if len(labeled_data) > 0:
        topic_counts = labeled_data['label'].value_counts()
        can_stratify = all(topic_counts >= 2)
        
        if can_stratify:
            train_labeled, val_labeled = train_test_split(
                labeled_data, test_size=0.2, stratify=labeled_data['label'], random_state=42
            )
            print(f"  {option_name}: ✓ Stratified split")
        else:
            train_labeled, val_labeled = train_test_split(
                labeled_data, test_size=0.2, random_state=42
            )
            print(f"  {option_name}: ⚠ Random split (some topics < 2 examples)")
        
        if len(unlabeled_data) > 0:
            train_data = pd.concat([train_labeled, unlabeled_data], ignore_index=True)
            print(f"      Added {len(unlabeled_data)} unlabeled to training")
        else:
            train_data = train_labeled
        
        val_data = val_labeled
    else:
        train_data = data
        val_data = data.head(0)
        print(f"  {option_name}: ⚠ No labeled data for validation")
    
    return train_data, val_data

# Split each option
train_opt1, val_opt1 = split_with_stratification(data_opt1, "Option 1")
train_opt2, val_opt2 = split_with_stratification(data_opt2, "Option 2")
train_opt3, val_opt3 = split_with_stratification(data_opt3, "Option 3")
train_opt4, val_opt4 = split_with_stratification(data_opt4, "Option 4")

print(f"\n✓ All dataset options prepared")


CREATING DATASET OPTIONS & TRAIN/VAL SPLIT

Step 1: Grouping data into options...
  Option 1 (Labeled only):              69 examples
  Option 2 (Labeled + Pseudo):         759 examples
  Option 3 (Labeled + Unlabeled):      276 examples
  Option 4 (All) ⭐ RECOMMENDED:       966 examples

Step 2: Splitting each option into train/val...
  Option 1: ✓ Stratified split
  Option 2: ✓ Stratified split
  Option 3: ✓ Stratified split
      Added 207 unlabeled to training
  Option 4: ✓ Stratified split
      Added 207 unlabeled to training

✓ All dataset options prepared


In [28]:
# ============================================================
# CELL 7.1: SETUP SBERT TRAINING ENVIRONMENT
# ============================================================

print(f"\n{'='*60}")
print("SETTING UP SBERT TRAINING ENVIRONMENT (v7)")
print(f"{'='*60}")

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader
    from transformers import (
        AutoModel,
        AutoTokenizer,
        AutoConfig,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding,
        EvalPrediction
    )
    from transformers.modeling_outputs import SequenceClassifierOutput
    from datasets import Dataset
    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        precision_recall_fscore_support,
        f1_score,
        classification_report
    )
    
    print("✓ All required libraries available")
    
    # Check GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    TRAINING_AVAILABLE = True
    
except ImportError as e:
    print(f"⚠ Missing library: {e}")
    print("  Install: pip install transformers datasets torch sklearn")
    TRAINING_AVAILABLE = False

# ============================================================
# SBERT ARCHITECTURE COMPONENTS
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("DEFINING SBERT ARCHITECTURE")
    print(f"{'='*60}")
    
    class MeanPooling(nn.Module):
        """Mean pooling over all tokens (weighted by attention mask)."""
        def forward(self, last_hidden_state, attention_mask):
            mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)  # [B,S,1]
            summed = (last_hidden_state * mask).sum(dim=1)  # [B,H]
            counts = mask.sum(dim=1).clamp(min=1e-9)  # [B,1]
            return summed / counts
    
    class SBERTClassifier(nn.Module):
        """SBERT-style classifier with mean pooling + configurable head."""
        def __init__(self, base_name: str, num_labels: int, use_multi_label: bool = False, dropout: float = 0.1):
            super().__init__()
            self.encoder = AutoModel.from_pretrained(base_name)
            hidden = self.encoder.config.hidden_size
            self.pool = MeanPooling()
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Linear(hidden, num_labels)
            
            # Store config
            self.config = AutoConfig.from_pretrained(base_name)
            self.config.num_labels = num_labels
            self.config.problem_type = "multi_label_classification" if use_multi_label else "single_label_classification"
            self.use_multi_label = use_multi_label
            
            self.id2label = None
            self.label2id = None
        
        def forward(self, input_ids=None, attention_mask=None, **kwargs):
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            sent = self.pool(out.last_hidden_state, attention_mask)
            sent = self.dropout(sent)
            logits = self.classifier(sent)
            return SequenceClassifierOutput(logits=logits)
    
    print("✓ SBERT architecture defined:")
    print("  - MeanPooling: Average over all tokens (vs [CLS] token)")
    print("  - SBERTClassifier: BERT encoder + mean pooling + classification head")
    print("  - Supports both single-label and multi-label classification")


SETTING UP SBERT TRAINING ENVIRONMENT (v7)
✓ All required libraries available

Device: cuda
  GPU: NVIDIA GeForce RTX 3050
  Memory: 8.59 GB

DEFINING SBERT ARCHITECTURE
✓ SBERT architecture defined:
  - MeanPooling: Average over all tokens (vs [CLS] token)
  - SBERTClassifier: BERT encoder + mean pooling + classification head
  - Supports both single-label and multi-label classification


In [ ]:
# ============================================================
# CELL 7.2: PREPARE DATA FOR SBERT + MULTI-LABEL TRAINING
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("PREPARING DATA FOR SBERT TRAINING")
    print(f"{'='*60}")
    
    # =====================
    # 1) PICK DATASET OPTION
    # =====================
    
    if 'training' not in CONFIG:
        CONFIG['training'] = {'dataset_option': 'option4'}
    
    dataset_option = CONFIG['training'].get('dataset_option', 'option4')
    
    if dataset_option == 'option1':
        train_dataset, val_dataset = train_opt1, val_opt1
    elif dataset_option == 'option2':
        train_dataset, val_dataset = train_opt2, val_opt2
    elif dataset_option == 'option3':
        train_dataset, val_dataset = train_opt3, val_opt3
    else:
        train_dataset, val_dataset = train_opt4, val_opt4
    
    print(f"\nUsing {dataset_option}:")
    print(f"  Train: {len(train_dataset)} examples")
    print(f"  Val:   {len(val_dataset)} examples")
    
    # =====================
    # 2) DETECT MULTI-LABEL MODE (check for cosine columns)
    # =====================
    
    train_cols = list(train_dataset.columns)
    cos_cols = [c for c in train_cols if isinstance(c, str) and c.startswith("cos_")]
    
    # Keep only columns that exist in both train and val
    cos_cols = [c for c in cos_cols if c in val_dataset.columns]
    
    USE_MULTI_LABEL = len(cos_cols) == len(label2id)
    
    # EXPLICIT MODE DETECTION BANNER
    print("\n" + "="*60)
    if USE_MULTI_LABEL:
        print("🎯 MULTI-LABEL MODE ENABLED")
        print(f"   Using {len(cos_cols)} cosine columns as soft targets")
        print(f"   Topics: {list(label2id.keys())}")
        print(f"   Cosine columns: {cos_cols}")
    else:
        print("🎯 SINGLE-LABEL MODE ENABLED")
        print(f"   Using primary_topic as hard target")
        print(f"   Topics: {list(label2id.keys())}")
        print(f"   Note: Found {len(cos_cols)} cosine columns (expected {len(label2id)} for multi-label)")
    print("="*60 + "\n")
    
    # =====================
    # 3) LOAD MODEL + TOKENIZER
    # =====================
    
    from pathlib import Path
    
    SBERT_MODEL_NAME = "GroNLP/bert-base-dutch-cased"  # Dutch-capable BERT
    
    if CONFIG.get('model', {}).get('use_pretrained') and CONFIG.get('paths', {}).get('pretrained_model_path'):
        model_path = CONFIG['paths']['pretrained_model_path']
        if Path(model_path).exists():
            model_name = model_path
            print(f"\n✓ Loading pretrained model from: {model_path}")
        else:
            model_name = SBERT_MODEL_NAME
            print(f"\n⚠ Pretrained path not found, using base: {SBERT_MODEL_NAME}")
    else:
        model_name = SBERT_MODEL_NAME
        print(f"\n✓ Using base model: {SBERT_MODEL_NAME}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = SBERTClassifier(
        base_name=model_name,
        num_labels=len(label2id),
        use_multi_label=USE_MULTI_LABEL,
        dropout=0.1
    )
    model.id2label = id2label
    model.label2id = label2id
    model.to(device)
    
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"  Architecture: SBERT with mean pooling")
    
    # =====================
    # 4) COMPUTE CLASS WEIGHTS (for imbalanced topics)
    # =====================
    
    train_labels = train_dataset[train_dataset['label_id'] != -1]['label_id'].values
    label_counts = np.bincount(train_labels, minlength=len(label2id))
    
    # Inverse frequency weighting with sqrt smoothing
    total = label_counts.sum()
    class_weights = np.sqrt(total / (label_counts + 1))  # +1 to avoid division by zero
    class_weights = class_weights / class_weights.mean()  # normalize
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
    
    print(f"\n✓ Computed class weights (sqrt inverse frequency):")
    for idx, (label, weight) in enumerate(zip(id2label.values(), class_weights)):
        print(f"  {label}: {weight:.3f} (n={label_counts[idx]})")
    
    # =====================
    # 5) PREPARE DATASETS
    # =====================
    
    def tokenize_function(examples):
        return tokenizer(examples['text'], truncation=True, max_length=512)
    
    # Filter out unlabeled data from training
    train_labeled = train_dataset[train_dataset['label_id'] != -1].copy()
    
    # Select columns based on mode
    if USE_MULTI_LABEL:
        # Multi-label: include all cos_* columns
        base_cols = ['text', 'label_id'] + cos_cols
        
        # Create multi-label targets from cosine scores (threshold at 0.5)
        def create_multilabel_targets(row):
            cos_scores = [row[c] if c in row and pd.notna(row[c]) else 0.0 for c in cos_cols]
            # Binary targets: 1 if cosine score > threshold, 0 otherwise
            threshold = 0.5
            return [1 if score >= threshold else 0 for score in cos_scores]
        
        # build multi-label numpy targets for train set
        threshold = 0.5
        train_label_array = np.zeros((len(train_labeled), len(label2id)), dtype=np.float32)
        for i, (idx, row) in enumerate(train_labeled.iterrows()):
            for j, col in enumerate(cos_cols):
                score = row[col] if col in row and pd.notna(row[col]) else 0.0
                train_label_array[i, j] = 1.0 if score >= threshold else 0.0

        # Create dataset with proper numpy arrays
        train_data_dict = {
            'text': train_labeled['text'].tolist(),
            'labels': train_label_array  # numpy array (n_samples, n_labels)
        }
        hf_train = Dataset.from_dict(train_data_dict)

        # prepare validation labels (keep val_dataset_copy created later in original flow)
        val_dataset_copy = val_dataset.copy()
        val_label_array = np.zeros((len(val_dataset_copy), len(label2id)), dtype=np.float32)
        for i, (_, row) in enumerate(val_dataset_copy.iterrows()):
            for j, col in enumerate(cos_cols):
                score = row[col] if col in row and pd.notna(row[col]) else 0.0
                val_label_array[i, j] = 1.0 if score >= threshold else 0.0
        val_data_dict = {'text': val_dataset_copy['text'].tolist(), 'labels': val_label_array}
        hf_val = Dataset.from_dict(val_data_dict)
    else:
        # Single-label: just use label_id
        hf_train = Dataset.from_pandas(train_labeled[['text', 'label_id']].reset_index(drop=True))
        hf_val = Dataset.from_pandas(val_dataset[['text', 'label_id']].reset_index(drop=True))
        hf_train = hf_train.rename_column('label_id', 'labels')
        hf_val = hf_val.rename_column('label_id', 'labels')
    
    # Tokenize
    hf_train = hf_train.map(tokenize_function, batched=True, remove_columns=['text'])
    hf_val = hf_val.map(tokenize_function, batched=True, remove_columns=['text'])
    
    # Set format
    hf_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    hf_val.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    

    
    print(f"\n✓ Data prepared:")
    print(f"  Train: {len(hf_train)} examples")
    print(f"  Val:   {len(hf_val)} examples")
    print(f"  Mode:  {'Multi-label' if USE_MULTI_LABEL else 'Single-label'}")

else:
    print("⚠ Skipping data preparation - transformers library not available")


In [30]:
import torch
from typing import Dict, List, Any

class PolicyTextDataCollator:
    """
    Custom data collator for policy text classification.
    Handles both single-label and multi-label classification with proper label preservation.
    """
    
    def __init__(self, tokenizer, padding=True, max_length=512):
        self.tokenizer = tokenizer
        self.padding = padding
        self.max_length = max_length
        
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """
        Collate batch of pre-tokenized features.
        
        For thesis: Ensures labels (policy/slavery/reparative themes) are preserved
        during batching for the trainer.
        """
        
        # Separate labels from input features
        labels = None
        has_labels = "labels" in features[0]
        
        if has_labels:
            # Extract labels and keep them separate
            labels = [f["labels"] for f in features]
            # Remove labels from features temporarily for tokenizer padding
            features_for_padding = []
            for f in features:
                feature_dict = {k: v for k, v in f.items() if k != "labels"}
                features_for_padding.append(feature_dict)
        else:
            features_for_padding = features
        
        # Pad the input features using tokenizer's padding method
        batch = self.tokenizer.pad(
            features_for_padding,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Add labels back to the batch
        if has_labels:
            # Stack labels into a batch tensor
            if isinstance(labels[0], torch.Tensor):
                batch["labels"] = torch.stack(labels)
            else:
                batch["labels"] = torch.tensor(labels)
        
        return batch

In [31]:
# ============================================================
# CELL 7.3: TRAIN SBERT MODEL WITH UNASSIGNED GATE + CALIBRATION
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("TRAINING SBERT MODEL (v7)")
    print(f"{'='*60}")
    
    # =====================
    # 1) UNASSIGNED GATE CONFIGURATION
    # =====================
    
    USE_UNASSIGNED_GATE = True
    UNASSIGNED_LABEL_CANDIDATES = ["Unassigned", "Uncategorised", "Uncategorized", "Other"]
    
    # Find unassigned label index
    UNASSIGNED_IDX = None
    for candidate in UNASSIGNED_LABEL_CANDIDATES:
        if candidate in label2id:
            UNASSIGNED_IDX = label2id[candidate]
            break
    
    if UNASSIGNED_IDX is not None:
        print(f"\n✓ Unassigned gate ENABLED")
        print(f"  Unassigned label: '{list(label2id.keys())[UNASSIGNED_IDX]}' (index {UNASSIGNED_IDX})")
        print(f"  Mode: Hard threshold (will be calibrated post-training)")
    else:
        USE_UNASSIGNED_GATE = False
        print(f"\n⚠ Unassigned gate DISABLED (no matching label found)")
        print(f"  Looked for: {UNASSIGNED_LABEL_CANDIDATES}")
    
    # =====================
    # 2) METRICS FUNCTION
    # =====================
    
    def compute_metrics(eval_pred):
        if isinstance(eval_pred, EvalPrediction):
            logits, labels = eval_pred.predictions, eval_pred.label_ids
        else:
            logits, labels = eval_pred
        
        if USE_MULTI_LABEL:
            # Multi-label metrics
            probs = torch.sigmoid(torch.tensor(logits)).numpy()
            preds = (probs >= 0.5).astype(int)
            labels_int = labels.astype(int)
            
            return {
                'accuracy': accuracy_score(labels_int, preds),
                'precision': precision_recall_fscore_support(labels_int, preds, average='weighted', zero_division=0)[0],
                'recall': precision_recall_fscore_support(labels_int, preds, average='weighted', zero_division=0)[1],
                'f1': f1_score(labels_int, preds, average='weighted', zero_division=0)
            }
        else:
            # Single-label metrics
            preds = np.argmax(logits, axis=1)
            precision, recall, f1, _ = precision_recall_fscore_support(
                labels, preds, average='weighted', zero_division=0
            )
            return {
                'accuracy': accuracy_score(labels, preds),
                'precision': precision,
                'recall': recall,
                'f1': f1
            }
    
    # =====================
    # 3) CUSTOM TRAINER WITH WEIGHTED LOSS
    # =====================
    
    class WeightedLossTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            # Extract labels safely
            if "labels" in inputs:
                labels = inputs.pop("labels")
            elif "label" in inputs:
                labels = inputs.pop("label")
            else:
                raise KeyError("Neither 'labels' nor 'label' found in inputs")
    
            # Forward pass through the model to obtain outputs and logits
            outputs = model(**inputs)
            # Transformers models usually expose logits as outputs.logits, otherwise fallback to first element
            logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
    
            # Ensure class_weights is on same device as logits
            if isinstance(class_weights, torch.Tensor):
                cw = class_weights.to(logits.device)
            else:
                cw = torch.tensor(class_weights, device=logits.device)
    
            if USE_MULTI_LABEL:
                # Multi-label: BCEWithLogitsLoss expects float labels and pos_weight per class
                labels = labels.float().to(logits.device)
                loss_fn = nn.BCEWithLogitsLoss(pos_weight=cw)
                loss = loss_fn(logits, labels)
            else:
                # Single-label: CrossEntropyLoss expects long labels and weight per class
                labels = labels.long().to(logits.device)
                loss_fn = nn.CrossEntropyLoss(weight=cw)
                loss = loss_fn(logits, labels)
    
            return (loss, outputs) if return_outputs else loss
    
    # =====================
    # 4) TRAINING ARGUMENTS
    # =====================
    
    model_output_dir = str(fs.folders['Model_finetuning']) if 'fs' in globals() else "./sbert_model"
    
    # Use CONFIG training params or defaults
    training_config = CONFIG.get('training', {})
    
    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=training_config.get('num_epochs', 5),
        per_device_train_batch_size=training_config.get('batch_size_train', 16),
        per_device_eval_batch_size=training_config.get('batch_size_eval', 32),
        learning_rate=training_config.get('learning_rate', 2e-5),
        weight_decay=training_config.get('weight_decay', 0.01),
        warmup_ratio=training_config.get('warmup_ratio', 0.1),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"{model_output_dir}/logs",
        logging_strategy="steps",
        logging_steps=50,
        report_to=None,
        fp16=torch.cuda.is_available(),
        seed=42,
        push_to_hub=False,
    )
    
    print(f"\nTraining configuration:")
    print(f"  Epochs: {training_args.num_train_epochs}")
    print(f"  Batch size (train): {training_args.per_device_train_batch_size}")
    print(f"  Batch size (eval): {training_args.per_device_eval_batch_size}")
    print(f"  Learning rate: {training_args.learning_rate}")
    print(f"  FP16: {training_args.fp16}")
    
    # =====================
    # 5) CREATE TRAINER AND TRAIN
    # =====================
    
    # Instead of: data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    # Use this custom collator:
    data_collator = PolicyTextDataCollator(
        tokenizer=tokenizer,
        padding=True,
        max_length=512
    )



    # Now create the trainer with the fixed collator
    trainer = WeightedLossTrainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=data_collator,  # Use the custom collator
        compute_metrics=compute_metrics,
    )

    
    print(f"\n{'='*60}")
    print(f"Starting training...")
    print(f"{'='*60}\n")
    
    train_result = trainer.train()
    
    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")
    
    # =====================
    # 6) EVALUATE
    # =====================
    
    eval_results = trainer.evaluate()
    
    print(f"\nValidation Results:")
    print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"  Precision: {eval_results['eval_precision']:.4f}")
    print(f"  Recall:    {eval_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")
    
    # =====================
    # 7) PER-CLASS THRESHOLD CALIBRATION (SBERT-style)
    # =====================
    
    print(f"\n{'='*60}")
    print("CALIBRATING PER-CLASS THRESHOLDS")
    print(f"{'='*60}")
    
    val_pred = trainer.predict(hf_val)
    val_logits = val_pred.predictions
    val_labels = np.array(hf_val["labels"])
    
    if USE_MULTI_LABEL:
        # Multi-label: find best threshold per class
        probs = torch.sigmoid(torch.tensor(val_logits)).numpy()
        
        def find_best_thresholds(y_true, probs, grid=np.linspace(0.05, 0.95, 19)):
            C = y_true.shape[1]
            best = [0.5] * C
            for c in range(C):
                y = y_true[:, c]
                if y.sum() == 0:
                    continue
                p = probs[:, c]
                best_f1, best_t = 0.0, 0.5
                for t in grid:
                    f1 = f1_score(y, (p >= t).astype(int), zero_division=0)
                    if f1 > best_f1:
                        best_f1, best_t = f1, t
                best[c] = float(best_t)
            return best
        
        thresholds = find_best_thresholds(val_labels.astype(int), probs)
        
        print(f"\nPer-class thresholds (optimized for F1):")
        for idx, (label, thr) in enumerate(zip(id2label.values(), thresholds)):
            print(f"  {label}: {thr:.3f}")
        
        # Save thresholds
        threshold_info = {
            "topic_names": list(id2label.values()),
            "thresholds": thresholds,
            "use_unassigned_gate": USE_UNASSIGNED_GATE,
            "unassigned_idx": UNASSIGNED_IDX,
            "mode": "multi_label"
        }
        
        if 'fs' in globals():
            fs.save_data(threshold_info, "sbert_thresholds", "Model_finetuning", "json")
            print("\n✓ Saved: Model_finetuning/sbert_thresholds.json")
    
    else:
        # Single-label: temperature scaling
        def nll_with_T(T):
            T = max(0.05, float(T))
            z = torch.tensor(val_logits) / T
            logp = torch.log_softmax(z, dim=1).numpy()
            return -float(np.mean([logp[i, val_labels[i]] for i in range(len(val_labels))]))
        
        T = 1.0
        for _ in range(20):
            cands = [max(0.05, T * f) for f in (0.5, 0.75, 1.0, 1.25, 1.5)]
            losses = [nll_with_T(c) for c in cands]
            T = cands[int(np.argmin(losses))]
        
        p_cal = torch.softmax(torch.tensor(val_logits) / T, dim=1).numpy()
        pmax_cal = p_cal.max(axis=1)
        
        # Tier thresholds (30% high, 30% medium, 40% low/irrelevant)
        hi_tau = float(np.quantile(pmax_cal, 0.70))
        med_tau = float(np.quantile(pmax_cal, 0.40))
        
        print(f"\nTemperature scaling:")
        print(f"  Optimal T: {T:.3f}")
        print(f"  High confidence threshold: {hi_tau:.3f}")
        print(f"  Medium confidence threshold: {med_tau:.3f}")
        
        calib_info = {
            "temperature_T": T,
            "tier_thresholds": {
                "HIGH": hi_tau,
                "MEDIUM": med_tau,
                "LOW_or_IRRELEVANT": 0.0
            },
            "mode": "single_label"
        }
        
        if 'fs' in globals():
            fs.save_data(calib_info, "sbert_temperature_and_tiers", "Model_finetuning", "json")
            print("\n✓ Saved: Model_finetuning/sbert_temperature_and_tiers.json")
    
    # =====================
    # 8) SAVE MODEL + METRICS
    # =====================
    
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)
    
    metrics = {
        "train_loss": float(train_result.training_loss),
        "train_runtime": train_result.metrics['train_runtime'],
        "eval_accuracy": eval_results['eval_accuracy'],
        "eval_precision": eval_results['eval_precision'],
        "eval_recall": eval_results['eval_recall'],
        "eval_f1": eval_results['eval_f1'],
        "eval_loss": eval_results['eval_loss'],
        "num_train_examples": len(hf_train),
        "num_eval_examples": len(hf_val),
        "num_epochs": training_args.num_train_epochs,
        "dataset_used": dataset_option,
        "architecture": "SBERT",
        "mode": "multi_label" if USE_MULTI_LABEL else "single_label",
        "unassigned_gate": USE_UNASSIGNED_GATE
    }
    
    if 'fs' in globals():
        fs.save_data(metrics, "sbert_training_metrics", "Model_finetuning", "json")
        fs.save_config("checkpoint7_sbert_trained")
        print("✓ Saved: Model_finetuning/sbert_training_metrics.json")
        print("✓ Checkpoint saved: checkpoint7_sbert_trained")
    
    print(f"\n{'='*60}")
    print("✓ SBERT TRAINING COMPLETE")
    print(f"{'='*60}")
    print(f"\nModel architecture: SBERT with mean pooling")
    print(f"Classification mode: {'Multi-label' if USE_MULTI_LABEL else 'Single-label'}")
    print(f"Unassigned gate: {'Enabled' if USE_UNASSIGNED_GATE else 'Disabled'}")
    print(f"Class weighting: Enabled (sqrt inverse frequency)")
    print(f"Per-class thresholds: {'Calibrated' if USE_MULTI_LABEL else 'Temperature scaled'}")

else:
    print("⚠ Skipping training - required libraries not available")


TRAINING SBERT MODEL (v7)

⚠ Unassigned gate DISABLED (no matching label found)
  Looked for: ['Unassigned', 'Uncategorised', 'Uncategorized', 'Other']

Training configuration:
  Epochs: 5
  Batch size (train): 16
  Batch size (eval): 32
  Learning rate: 2e-05
  FP16: True

Starting training...



  0%|          | 0/190 [00:00<?, ?it/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


KeyError: "Neither 'labels' nor 'label' found in inputs"

In [32]:
# Debug: Check what columns your dataset has
print("Training dataset columns:", hf_train.column_names if hasattr(hf_train, 'column_names') else hf_train.features.keys())
print("Validation dataset columns:", hf_val.column_names if hasattr(hf_val, 'column_names') else hf_val.features.keys())

# Check a sample from the dataset
print("\nSample from training dataset:")
print(hf_train[0])

Training dataset columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Validation dataset columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']

Sample from training dataset:
{'labels': tensor([0., 0., 0.]), 'input_ids': tensor([    1,   423,    81,    41, 23565, 23310,   422,    13,    37,    13,
         7682, 24257,   131,    11,  2058,     0, 13644,  6754,    11, 16203,
        10647, 12959, 25108, 10537,   130,     0,    11, 10669, 13644, 10537,
           61,  5122, 25138, 26028,    62, 12395,     0,    11,  9265, 16779,
        13644, 16760,  7601,    12,  3580, 14491,     5,     9,   772,    25,
           35,    13, 10537,  3445,   120,    11,   229, 23107,    10,    26,
           37,    13,  7682, 24257,   131,    11,  6865, 21901,   117, 20722,
        10537, 11269, 10537,   130, 18991,     9,   772,    25,    35,    13,
        10537,  3445,   120,    11,   229, 23107,    10,    13,   423,    13,
           37,    13,  7682, 24257,   131,    1

✅ **CHECKPOINT 6 COMPLETE** - Training data prepared

**Resume**: Load train/val CSVs from `Model_finetuning/`

---
# CHECKPOINT 7: Model Training (BERTJE)
---

Fine-tune Dutch BERT on labeled data.

⚠️ **Note**: This requires `transformers` library and GPU for efficient training.

In [117]:
# ============================================================
# CELL 7.1: SETUP TRAINING
# ============================================================

try:
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding
    )
    from datasets import Dataset
    import torch
    
    print("✓ Transformers library available")
    
    # Check GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    
    TRAINING_AVAILABLE = True
    
except ImportError as e:
    print("⚠ Transformers library not available")
    print("  Install: pip install transformers datasets torch")
    TRAINING_AVAILABLE = False

✓ Transformers library available

Device: cuda
  GPU: NVIDIA GeForce RTX 3050


In [ ]:
# ============================================================
# CELL 7.2: LOAD & PREPARE DATA  (adds cos_* columns for soft labels)
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("PREPARING TRAINING DATA")
    print(f"{'='*60}")
    
    # -----------------------------[ 1) PICK DATASET ]-----------------------------
    dataset_option = CONFIG['training']['dataset_option']
    if dataset_option == 'option1':
        train_dataset = train_opt1
        val_dataset   = val_opt1
    elif dataset_option == 'option2':
        train_dataset = train_opt2
        val_dataset   = val_opt2
    elif dataset_option == 'option3':
        train_dataset = train_opt3
        val_dataset   = val_opt3
    else:
        train_dataset = train_opt4
        val_dataset   = val_opt4
    
    print(f"\nUsing {dataset_option}: {len(train_dataset)} examples")
    
    # -----------------------------[ 2) DETECT TRAINING MODE ]---------------------
    # Auto-detect columns that start with 'cos_' to determine training mode
    import numpy as np
    from datasets import Dataset
    
    train_cols = list(train_dataset.columns)
    default_cos_cols = [c for c in train_cols if isinstance(c, str) and c.startswith("cos_")]
    cos_cfg = CONFIG.get('cosine', {}) if isinstance(CONFIG.get('cosine', {}), dict) else {}
    cos_cols = cos_cfg.get('columns', default_cos_cols)
    
    # Keep only numeric cosine columns that exist in both splits
    cos_cols = [c for c in cos_cols if c in train_dataset.columns and c in val_dataset.columns]
    # Optional: ensure they are numeric
    for c in list(cos_cols):
        try:
            _ = np.asarray(train_dataset[c].astype(float))
            _ = np.asarray(val_dataset[c].astype(float))
        except Exception:
            print(f"  ⚠ Skipping non-numeric cosine column: {c}")
            cos_cols.remove(c)
    
    # EXPLICIT MODE DETECTION BANNER
    is_multilabel = len(cos_cols) == len(label2id)
    
    print("\n" + "="*60)
    if is_multilabel:
        print("🎯 MULTI-LABEL MODE ENABLED")
        print(f"   Using {len(cos_cols)} cosine columns as soft targets")
        print(f"   Topics: {list(label2id.keys())}")
        print(f"   Cosine columns: {cos_cols}")
    else:
        print("🎯 SINGLE-LABEL MODE ENABLED")
        print(f"   Using primary_topic as hard target")
        print(f"   Topics: {list(label2id.keys())}")
        if len(cos_cols) > 0:
            print(f"   Note: Found {len(cos_cols)} cosine columns (expected {len(label2id)} for multi-label)")
        else:
            print(f"   Note: No cosine columns found")
    print("="*60 + "\n")
    
    # -----------------------------[ 3) LOAD MODEL + TOKENIZER ]-------------------
    # CRITICAL: Use SBERT architecture for consistency with Checkpoint 5 scoring
    from pathlib import Path
    from transformers import AutoTokenizer, DataCollatorWithPadding
    import torch.nn as nn
    
    if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
        model_path = CONFIG['paths']['pretrained_model_path']
        if Path(model_path).exists():
            model_name = model_path
            print(f"✓ Loading pretrained model from: {model_path}")
        else:
            model_name = "GroNLP/bert-base-dutch-cased"
            print(f"⚠ Pretrained path not found, using base model")
    else:
        model_name = "GroNLP/bert-base-dutch-cased"
        print(f"✓ Using base model: GroNLP/bert-base-dutch-cased")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Use SBERT architecture with mean pooling (same as Checkpoint 5)
    # This ensures consistency between scoring and training
    model = SBERTClassifier(
        base_name=model_name,
        num_labels=len(label2id),
        use_multi_label=is_multilabel,
        dropout=0.1
    )
    model.id2label = id2label
    model.label2id = label2id
    model.to(device)
    
    print(f"\n✓ Model loaded: {model_name}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"  Architecture: SBERT with mean pooling (consistent with Checkpoint 5)")
    
    # -----------------------------[ 4) TOKENIZER FN ]-----------------------------
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding=False,
            truncation=True,
            max_length=512
        )
    
    # -----------------------------[ 5) BUILD HF DATASETS ]------------------------
    # Include labels and all detected cos_* columns alongside tokenized inputs
    # Note: do not drop cos_* when mapping or set_format is applied
    base_train_cols = ['text', 'label_id'] + cos_cols
    base_val_cols   = ['text', 'label_id'] + cos_cols
    
    # guard against missing columns due to prior filtering
    base_train_cols = [c for c in base_train_cols if c in train_dataset.columns]
    base_val_cols   = [c for c in base_val_cols   if c in val_dataset.columns]
    
    train_labeled = train_dataset[train_dataset['label_id'] != -1].copy()
    hf_train = Dataset.from_pandas(train_labeled[base_train_cols].reset_index(drop=True))
    hf_val   = Dataset.from_pandas(val_dataset[base_val_cols].reset_index(drop=True))
    
    # map tokenizer (do not remove cosine columns)
    hf_train = hf_train.map(tokenize_function, batched=True, remove_columns=['text'])
    hf_val   = hf_val.map(  tokenize_function, batched=True, remove_columns=['text'])
    
    # rename label column
    hf_train = hf_train.rename_column('label_id', 'labels')
    hf_val   = hf_val.rename_column('label_id', 'labels')
    
    # set tensor format, include cos_* so collator can see them
    tensor_cols_train = ['input_ids', 'attention_mask', 'labels'] + cos_cols
    tensor_cols_val   = ['input_ids', 'attention_mask', 'labels'] + cos_cols
    hf_train.set_format(type='torch', columns=tensor_cols_train)
    hf_val.set_format(type='torch',   columns=tensor_cols_val)
    
    # data collator stays the same here, CELL 7.3 will wrap it when needed
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    print(f"\n✓ Data prepared")
    print(f"  Train: {len(hf_train)}, Val: {len(hf_val)}")
    print(f"  Mode: {'Multi-label' if is_multilabel else 'Single-label'}")
    if len(cos_cols) > 0:
        print(f"  Included cosine columns in HF datasets: {cos_cols}")
    else:
        print(f"  No cosine columns included (none detected)")


In [128]:
# ============================================================
# CELL 7.3: TRAIN MODEL  (BERTje with cosine soft labels + reject gate)
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("TRAINING MODEL  (BERTje with cosine soft labels + reject gate)")
    print(f"{'='*60}")

    # -----------------------------[ IMPORTS ]-----------------------------
    import os, json
    import numpy as np
    import torch
    import torch.nn.functional as F
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    from transformers import TrainingArguments, Trainer
    from transformers import DataCollatorWithPadding

    # -----------------------------[ 0) METRICS: unchanged ]----------------
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )
        return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    # -----------------------------[ 1) COSINE SOFT-LABEL SETUP ]----------
    print("\n[SETUP] Detecting cosine columns and soft-label settings...")
    num_labels = int(getattr(model.config, "num_labels", 3))

    train_cols = hf_train.column_names
    default_cos_cols = [c for c in train_cols if c.startswith("cos_")]
    cos_cfg = CONFIG.get('cosine', {}) if isinstance(CONFIG.get('cosine', {}), dict) else {}
    cos_cols = cos_cfg.get('columns', default_cos_cols)

    if len(cos_cols) != num_labels:
        print(f"  ⚠ Expected {num_labels} cosine columns, found {len(cos_cols)}: {cos_cols}. Using hard labels.")
        cos_cols = []  # disable soft-label path

    cos_tau = float(cos_cfg.get('softmax_tau', 0.5))         # temperature for soft labels from cosine
    other_threshold = float(cos_cfg.get('other_threshold', 0.40))  # low max-cos => likely irrelevant
    other_weight = float(cos_cfg.get('other_weight', 0.30))        # weight floor for likely-irrelevant

    # -----------------------------[ 2) DATA COLLATOR (passes cos_*) ]------
    if cos_cols:
        base_collator = data_collator if data_collator is not None else DataLakollocatorWithPadding(tokenizer)

        class CollatorWithCosine:
            def __init__(self, base, cos_columns):
                self.base = base
                self.cos_columns = list(cos_columns)
            def __call__(self, features):
                # keep cosine values before base collator filters anything
                cos_buf = {c: [float(f.get(c, 0.0)) for f in features] for c in self.cos_columns}
                batch = self.base(features)  # turns text parts into tensors
                # add cosine tensors
                for c, vals in cos_buf.items():
                    batch[c] = torch.tensor(vals, dtype=torch.float)
                return batch

        effective_collator = CollatorWithCosine(base_collator, cos_cols)
        print(f"  ✓ Using soft labels from cosine columns: {cos_cols}")
        print(f"  ✓ Cosine softmax tau: {cos_tau}, irrelevant gate during training (thr={other_threshold}, floor={other_weight})")
    else:
        effective_collator = data_collator
        print("  ✓ Proceeding with hard-label training (no cosine columns used)")

    # -----------------------------[ 3) CUSTOM TRAINER (fix: accept num_items_in_batch) ]---
    class SoftLabelTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.get("labels")
            # strip out non-model keys
            model_inputs = {k: v for k, v in inputs.items() if k not in (["labels"] + list(cos_cols))}
            outputs = model(**model_inputs)  # standard forward
            logits = outputs.logits  # [B, C]

            if cos_cols:
                cos_stack = torch.stack([inputs[c].float() for c in cos_cols], dim=1)  # [B, C]
                # temperature softmax on cosine scores -> soft targets
                cos_norm = (cos_stack / max(cos_tau, 1e-6)).softmax(dim=1)            # [B, C]
                # per-example weights (downweight likely-irrelevant)
                w = cos_stack.max(dim=1).values
                w = torch.where(w < other_threshold, torch.full_like(w, other_weight), w)
                w = torch.clamp(w, min=1e-3)
                # cross-entropy with soft labels
                logp = F.log_softmax(logits, dim=1)                                   # [B, C]
                loss_vec = -(cos_norm * logp).sum(dim=1)                               # [B]
                loss = (loss_vec * w).sum() / w.sum()
            else:
                loss = F.cross_entropy(logits, labels)

            return (loss, outputs) if return_outputs else loss

    # -----------------------------[ 4) TRAINING ARGS: add remove_unused_columns=False ]----
    model_output_dir = str(fs.folders['Model_finetuning'])

    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=CONFIG['training']['num_epochs'],
        per_device_train_batch_size=CONFIG['training']['batch_size_train'],
        per_device_eval_batch_size=CONFIG['training']['batch_size_eval'],
        learning_rate=CONFIG['training']['learning_rate'],
        weight_decay=CONFIG['training']['weight_decay'],
        warmup_ratio=CONFIG['training']['warmup_ratio'],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=str(fs.folders['Model_finetuning'] / "logs"),
        logging_strategy="steps",
        logging_steps=50,
        report_to=None,
        fp16=torch.cuda.is_available(),
        seed=42,
        push_to_hub=False,
        remove_unused_columns=False,  # <<< IMPORTANT so cos_* reach the collator
    )

    # -----------------------------[ 5) TRAIN ]-----------------------------
    trainer = SoftLabelTrainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=effective_collator,
        compute_metrics=compute_metrics,
    )

    print(f"\nStarting training for {CONFIG['training']['num_epochs']} epochs...")
    train_result = trainer.train()

    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")

    # -----------------------------[ 6) SAVE MODEL + TOKENIZER: unchanged ]-
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)

    # -----------------------------[ 7) EVALUATE: unchanged ]---------------
    eval_results = trainer.evaluate()

    print(f"\nValidation Results:")
    print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"  Precision: {eval_results['eval_precision']:.4f}")
    print(f"  Recall:    {eval_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")

    # -----------------------------[ 8) NEW: save soft-label info ]---------
    soft_info = {
        "used_cosine_columns": cos_cols,
        "cosine_softmax_tau": cos_tau,
        "other_threshold": other_threshold,
        "other_weight_floor": other_weight,
        "num_labels": int(num_labels),
        "used_soft_labels": bool(cos_cols)
    }
    fs.save_data(soft_info, "softlabel_training_info", "Model_finetuning", "json")
    print("✓ Saved: Model_finetuning/softlabel_training_info.json")

    # -----------------------------[ 9) NEW: calibrate reject tiers ]-------
    print("\n[CALIBRATION] Fitting temperature and computing tier thresholds...")
    val_pred = trainer.predict(hf_val)
    val_logits = val_pred.predictions
    val_labels = np.array(hf_val["labels"])

    def nll_with_T(T):
        T = max(0.05, float(T))
        z = torch.tensor(val_logits) / T
        logp = torch.log_softmax(z, dim=1).numpy()
        return -float(np.mean([logp[i, val_labels[i]] for i in range(len(val_labels))]))

    T = 1.0
    for _ in range(20):  # cheap line search
        cands = [max(0.05, T * f) for f in (0.5, 0.75, 1.0, 1.25, 1.5)]
        losses = [nll_with_T(c) for c in cands]
        T = cands[int(np.argmin(losses))]

    p_cal = torch.softmax(torch.tensor(val_logits) / T, dim=1).numpy()
    pmax_cal = p_cal.max(axis=1)

    # Target shares for tiers (simple defaults; override in CONFIG['cosine'] if you want)
    hi_share  = float(cos_cfg.get('target_high_share', 0.30))
    med_share = float(cos_cfg.get('target_med_share', 0.30))

    hi_tau  = float(np.quantile(pmax_cal, 1.0 - min(max(hi_share, 0.01), 0.95)))
    med_tau = float(np.quantile(pmax_cal, 1.0 - min(max(hi_share + med_share, 0.02), 0.98)))

    calib = {
        "temperature_T": T,
        "tier_thresholds": {
            "HIGH":   hi_tau,
            "MEDIUM": med_tau,
            "LOW_or_IRRELEVANT": 0.0
        },
        "note": "At inference: softmax(logits / T). If pmax≥HIGH→HIGH, if MEDIUM≤pmax<HIGH→MEDIUM, else LOW/Irrelevant."
    }
    fs.save_data(calib, "bert_temperature_and_tiers", "Model_finetuning", "json")
    print("✓ Saved: Model_finetuning/bert_temperature_and_tiers.json")

    # -----------------------------[ 10) SAVE METRICS + CHECKPOINT: same ]--
    metrics = {
        "train_loss": train_result.training_loss,
        "train_runtime": train_result.metrics['train_runtime'],
        "eval_accuracy": eval_results['eval_accuracy'],
        "eval_precision": eval_results['eval_precision'],
        "eval_recall": eval_results['eval_recall'],
        "eval_f1": eval_results['eval_f1'],
        "eval_loss": eval_results['eval_loss'],
        "num_train_examples": len(hf_train),
        "num_eval_examples": len(hf_val),
        "num_epochs": CONFIG['training']['num_epochs'],
        "dataset_used": dataset_option,
    }
    fs.save_data(metrics, "training_metrics", "Model_finetuning", "json")
    fs.save_config("checkpoint7_trained")

    print(f"\n✓ Model and metrics saved")
    print("✓ Soft labels used" if cos_cols else "✓ Hard labels used (no cosine columns detected)")

else:
    print("⚠ Skipping training - transformers library not available")



TRAINING MODEL  (BERTje with cosine soft labels + reject gate)

[SETUP] Detecting cosine columns and soft-label settings...
  ✓ Using soft labels from cosine columns: ['cos_Colonialism', 'cos_Historical slavery', 'cos_Modern racism& inequality']
  ✓ Cosine softmax tau: 0.5, irrelevant gate during training (thr=0.4, floor=0.3)

Starting training for 5 epochs...


  0%|          | 0/190 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': nan, 'eval_accuracy': 0.2894736842105263, 'eval_precision': 0.30240252468695916, 'eval_recall': 0.2894736842105263, 'eval_f1': 0.175854108956602, 'eval_runtime': 5.3526, 'eval_samples_per_second': 28.397, 'eval_steps_per_second': 0.934, 'epoch': 1.0}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 0.0, 'epoch': 1.32}


  0%|          | 0/5 [00:00<?, ?it/s]

...

{'eval_loss': nan, 'eval_accuracy': 0.2894736842105263, 'eval_precision': 0.30240252468695916, 'eval_recall': 0.2894736842105263, 'eval_f1': 0.175854108956602, 'eval_runtime': 4.9896, 'eval_samples_per_second': 30.464, 'eval_steps_per_second': 1.002, 'epoch': 5.0}
{'train_runtime': 550.8827, 'train_samples_per_second': 5.509, 'train_steps_per_second': 0.345, 'train_loss': 0.0, 'epoch': 5.0}

TRAINING COMPLETE


  0%|          | 0/5 [00:00<?, ?it/s]


Validation Results:
  Accuracy:  0.2895
  Precision: 0.3024
  Recall:    0.2895
  F1 Score:  0.1759
✓ Saved: Model_finetuning/softlabel_training_info.json
✓ Saved: Model_finetuning/softlabel_training_info.json

[CALIBRATION] Fitting temperature and computing tier thresholds...


  0%|          | 0/5 [00:00<?, ?it/s]

✓ Saved: Model_finetuning/bert_temperature_and_tiers.json
✓ Saved: Model_finetuning/bert_temperature_and_tiers.json
✓ Saved: Model_finetuning/training_metrics.json
✓ Config saved: config_checkpoint7_trained_20251030_182737.json

✓ Model and metrics saved
✓ Soft labels used


In [22]:
# ============================================================
# CELL 7.3: TRAIN MODEL
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("TRAINING MODEL")
    print(f"{'='*60}")
    
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    # Training arguments
    model_output_dir = str(fs.folders['Model_finetuning'])
    
    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=CONFIG['training']['num_epochs'],
        per_device_train_batch_size=CONFIG['training']['batch_size_train'],
        per_device_eval_batch_size=CONFIG['training']['batch_size_eval'],
        learning_rate=CONFIG['training']['learning_rate'],
        weight_decay=CONFIG['training']['weight_decay'],
        warmup_ratio=CONFIG['training']['warmup_ratio'],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=str(fs.folders['Model_finetuning'] / "logs"),
        logging_strategy="steps",
        logging_steps=50,
        report_to=None,
        fp16=torch.cuda.is_available(),
        seed=42,
        push_to_hub=False,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    print(f"\nStarting training for {CONFIG['training']['num_epochs']} epochs...")
    train_result = trainer.train()
    
    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")
    
    # Save model
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)
    
    # Evaluate
    eval_results = trainer.evaluate()
    
    print(f"\nValidation Results:")
    print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"  Precision: {eval_results['eval_precision']:.4f}")
    print(f"  Recall:    {eval_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")
    
    # Save metrics
    metrics = {
        "train_loss": train_result.training_loss,
        "train_runtime": train_result.metrics['train_runtime'],
        "eval_accuracy": eval_results['eval_accuracy'],
        "eval_precision": eval_results['eval_precision'],
        "eval_recall": eval_results['eval_recall'],
        "eval_f1": eval_results['eval_f1'],
        "eval_loss": eval_results['eval_loss'],
        "num_train_examples": len(hf_train),
        "num_eval_examples": len(hf_val),
        "num_epochs": CONFIG['training']['num_epochs'],
        "dataset_used": dataset_option,
    }
    
    fs.save_data(metrics, "training_metrics", "Model_finetuning", "json")
    fs.save_config("checkpoint7_trained")
    
    print(f"\n✓ Model and metrics saved")
else:
    print("⚠ Skipping training - transformers library not available")


TRAINING MODEL

Starting training for 3 epochs...


  0%|          | 0/996 [00:00<?, ?it/s]

{'loss': 0.4401, 'grad_norm': 11.133905410766602, 'learning_rate': 9.800000000000001e-06, 'epoch': 0.15}
{'loss': 0.2769, 'grad_norm': 7.9429707527160645, 'learning_rate': 1.9600000000000002e-05, 'epoch': 0.3}
{'loss': 0.3182, 'grad_norm': 17.560598373413086, 'learning_rate': 1.8950892857142858e-05, 'epoch': 0.45}
{'loss': 0.2684, 'grad_norm': 23.28594398498535, 'learning_rate': 1.783482142857143e-05, 'epoch': 0.6}
{'loss': 0.3301, 'grad_norm': 11.638723373413086, 'learning_rate': 1.671875e-05, 'epoch': 0.75}
{'loss': 0.2435, 'grad_norm': 29.42177391052246, 'learning_rate': 1.5602678571428574e-05, 'epoch': 0.9}


  0%|          | 0/42 [00:00<?, ?it/s]

{'eval_loss': 0.22826729714870453, 'eval_accuracy': 0.9125188536953243, 'eval_precision': 0.9173267109504935, 'eval_recall': 0.9125188536953243, 'eval_f1': 0.9130577753926096, 'eval_runtime': 11.3351, 'eval_samples_per_second': 116.981, 'eval_steps_per_second': 3.705, 'epoch': 1.0}
{'loss': 0.2093, 'grad_norm': 54.80517578125, 'learning_rate': 1.4486607142857143e-05, 'epoch': 1.05}
{'loss': 0.1379, 'grad_norm': 28.29879379272461, 'learning_rate': 1.3370535714285714e-05, 'epoch': 1.2}
{'loss': 0.2204, 'grad_norm': 4.803442001342773, 'learning_rate': 1.2254464285714287e-05, 'epoch': 1.36}
{'loss': 0.1828, 'grad_norm': 5.236793041229248, 'learning_rate': 1.113839285714286e-05, 'epoch': 1.51}
{'loss': 0.1246, 'grad_norm': 0.43263348937034607, 'learning_rate': 1.0022321428571429e-05, 'epoch': 1.66}
{'loss': 0.1378, 'grad_norm': 13.823603630065918, 'learning_rate': 8.906250000000001e-06, 'epoch': 1.81}
{'loss': 0.1765, 'grad_norm': 6.028990745544434, 'learning_rate': 7.790178571428572e-06, '

...

{'eval_loss': 0.21923618018627167, 'eval_accuracy': 0.9215686274509803, 'eval_precision': 0.9217974529218909, 'eval_recall': 0.9215686274509803, 'eval_f1': 0.9206193430472789, 'eval_runtime': 11.2005, 'eval_samples_per_second': 118.388, 'eval_steps_per_second': 3.75, 'epoch': 2.0}
{'loss': 0.0933, 'grad_norm': 2.015324115753174, 'learning_rate': 6.674107142857143e-06, 'epoch': 2.11}
{'loss': 0.0775, 'grad_norm': 0.18117284774780273, 'learning_rate': 5.558035714285714e-06, 'epoch': 2.26}
{'loss': 0.0724, 'grad_norm': 31.171051025390625, 'learning_rate': 4.441964285714286e-06, 'epoch': 2.41}
{'loss': 0.0637, 'grad_norm': 0.0270305797457695, 'learning_rate': 3.3258928571428572e-06, 'epoch': 2.56}
{'loss': 0.0477, 'grad_norm': 46.910865783691406, 'learning_rate': 2.2321428571428573e-06, 'epoch': 2.71}
{'loss': 0.0619, 'grad_norm': 10.829289436340332, 'learning_rate': 1.1160714285714287e-06, 'epoch': 2.86}


  0%|          | 0/42 [00:00<?, ?it/s]

{'eval_loss': 0.2921321988105774, 'eval_accuracy': 0.9230769230769231, 'eval_precision': 0.9227142276022363, 'eval_recall': 0.9230769230769231, 'eval_f1': 0.9225774749942328, 'eval_runtime': 11.1783, 'eval_samples_per_second': 118.622, 'eval_steps_per_second': 3.757, 'epoch': 3.0}
{'train_runtime': 556.5764, 'train_samples_per_second': 28.589, 'train_steps_per_second': 1.79, 'train_loss': 0.17809077439059215, 'epoch': 3.0}

TRAINING COMPLETE


  0%|          | 0/42 [00:00<?, ?it/s]


Validation Results:
  Accuracy:  0.9231
  Precision: 0.9227
  Recall:    0.9231
  F1 Score:  0.9226
✓ Saved: Model_finetuning/training_metrics.json
✓ Config saved: config_checkpoint7_trained_20251030_150714.json

✓ Model and metrics saved


✅ **CHECKPOINT 7 COMPLETE** - Model trained and saved

**Resume**: Load model from `Model_finetuning/`

✅ **CHECKPOINT 9 COMPLETE** - Visualizations generated

**All checkpoints complete!** Check `Visuals/` folder for interactive plots.

---
# Workflow Complete! 🎉
---

## Summary

All checkpoints have been executed:

✅ **CHECKPOINT 0**: Setup & Configuration
✅ **CHECKPOINT 1**: Text Processing
✅ **CHECKPOINT 2**: Vocabulary Building
✅ **CHECKPOINT 3**: Dictionary Expansion
✅ **CHECKPOINT 4**: Topic Vectors
✅ **CHECKPOINT 5**: Chunk Scoring
✅ **CHECKPOINT 6**: Training Data Prep
✅ **CHECKPOINT 7**: Model Training
✅ **CHECKPOINT 8**: BERTJE Labeling
✅ **CHECKPOINT 9**: Visualizations

## Output Location

All outputs saved to: `{workflow_root}`

```
{ModelType}-{Topic}_{Date}_{Version}/
├── config/              # Config snapshots at each checkpoint
├── Dictionary/          # Input, expanded, curated dictionaries
│   └── Dictionary_suggestions/
├── Model_finetuning/    # Trained model + metrics
├── Cosine_labeling/     # Confidence-classified scores
├── Bertje_labeling/     # Model predictions
├── Visuals/             # Interactive HTML visualizations
└── Other_data/          # Chunks, vocabulary, topic vectors
```

## Next Steps

1. **Review Results**: Check visualizations in `Visuals/`
2. **Analyze Model**: Review training metrics
3. **Use Model**: Load trained model for predictions
4. **Iterate**: Adjust config and re-run from any checkpoint

## Using the Trained Model

To use this model in a new workflow:

```python
CONFIG['model']['use_pretrained'] = True
CONFIG['paths']['pretrained_model_path'] = 'path/to/Model_finetuning'
CONFIG['workflow']['model_type'] = 'Finetuned_{Source}'
```

See `WORKFLOW_GUIDE_v3.md` for complete documentation!